## Running Two Cluster Scenario
### Pooled SIR instead of monthly:

**Start Date:** 1 Sep, 2026

**Last Modified:** 3 Sep, 2026

In [1]:
"""Population-limited spatially varying temporal trend (SVTT) scan.

For every candidate spatial zone Z and temporal basis B(t), the statistic
compares

    H0: log(mu_in,t)  = log(E_in,t)  + alpha_in  + beta' B(t)
        log(mu_out,t) = log(E_out,t) + alpha_out + beta' B(t)

with

    H1: log(mu_in,t)  = log(E_in,t)  + alpha_in  + beta_in'  B(t)
        log(mu_out,t) = log(E_out,t) + alpha_out + beta_out' B(t).

Thus stable inside/outside level differences are nuisance intercepts; the scan
targets a difference in temporal slope/shape. Expected counts use one reference
rate per complete dataset, pooled across all locations and times:

    R0 = sum_i,t Y_it / sum_i,t P_it
    E_it = P_it * R0,  SIR_it = Y_it / E_it.

Counts are fitted as Poisson responses with log(E_it) offsets; SIR is exported
for interpretation. R0 is estimated once per dataset and is not a time-specific
rate. The conditional multinomial bootstrap fixes observed time totals and
uses fitted county intercepts, not equal county probabilities. The common
null time effect cancels on conditioning. This is a fitted-null bootstrap;
it does not condition on realized county totals or provide an exact test.

Monte Carlo datasets are drawn
under the no-interaction null (a common temporal trajectory) and the complete
maximum scan is recomputed for every replicate.

Candidate zones are limited by population at risk:

    100 * population(Z) / population(all locations) <= max_size_percent.

They are not limited by the percentage of location counts.

Portable folder layout
----------------------
Keep this file in the downloaded ``SVTT`` folder alongside
``nm_county_coordinates.txt`` and the data folders.  No computer-specific
absolute path is required.  If the project is stored elsewhere, set the
``SVTT_ROOT`` environment variable or pass ``coord_path`` explicitly when
constructing :class:`SVTT_Scan`.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal, Any, Mapping, Sequence
import ast
import gc
import os
import re
import traceback

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.spatial.distance import cdist
from sklearn.metrics import adjusted_rand_score, rand_score


# Increment this whenever a change makes old batch checkpoints statistically
# incompatible.  In particular, versions before this one limited candidate
# windows by the NUMBER of locations rather than by population at risk.
IMPLEMENTATION_VERSION = "2026-10-01-fixed-R0-population-v1"
EXPECTED_COUNT_DEFINITION = "population_it * pooled_R0"
NULL_CALIBRATION = "conditional_multinomial_county_intercepts"
WINDOW_SIZE_MEASURE = "mean_population_share"


def find_svtt_root(start: str | Path | None = None) -> Path:
    """Locate the portable SVTT project folder.

    Search order:
      1. ``SVTT_ROOT`` environment variable;
      2. an explicitly supplied ``start`` location;
      3. the directory containing this Python file;
      4. the current working directory;
      5. those locations' parents and nearby ``SVTT`` subfolders.

    The coordinate file is used as the project marker.  If it has not yet
    been copied into the project, the function returns the script directory
    (or the current directory in a notebook); ``SVTT_Scan`` then raises a
    clear error explaining what is missing.
    """

    marker_name = "nm_county_coordinates.txt"
    candidates: list[Path] = []

    root_override = os.environ.get("SVTT_ROOT")
    if root_override:
        candidates.append(Path(root_override).expanduser())

    if start is not None:
        start_path = Path(start).expanduser()
        candidates.append(
            start_path.parent if start_path.is_file() else start_path
        )

    if "__file__" in globals():
        candidates.append(Path(__file__).resolve().parent)

    candidates.append(Path.cwd())

    expanded_candidates: list[Path] = []
    seen: set[Path] = set()

    for initial in candidates:
        initial = initial.resolve()
        for candidate in (initial, *initial.parents):
            for possible_root in (candidate, candidate / "SVTT"):
                possible_root = possible_root.resolve()
                if possible_root not in seen:
                    seen.add(possible_root)
                    expanded_candidates.append(possible_root)

    for candidate in expanded_candidates:
        if (candidate / marker_name).is_file():
            return candidate

    if candidates:
        return candidates[0].resolve()

    return Path.cwd().resolve()


SVTT_ROOT = find_svtt_root()
DEFAULT_COORD_PATH = SVTT_ROOT / "nm_county_coordinates.txt"

@dataclass(frozen=True)
class MethodSpec:
    """
    Temporal basis specification.

    Polynomial:
        p is the polynomial degree and number of basis columns.

    Restricted cubic spline:
        p is the number of basis columns.
        The number of internal knots is p - 1.
    """

    name: str
    basis: Literal["poly", "spline"]
    p: int

    def __post_init__(self):

        if self.basis not in {"poly", "spline"}:
            raise ValueError(
                "basis must be either 'poly' or 'spline'."
            )

        if not isinstance(self.p, (int, np.integer)):
            raise TypeError(
                "p must be an integer."
            )

        if self.p < 1:
            raise ValueError(
                "p must be at least 1."
            )



DEFAULT_METHODS: tuple[MethodSpec, ...] = (

    MethodSpec(
        name="linear",
        basis="poly",
        p=1
    ),

    MethodSpec(
        name="quadratic",
        basis="poly",
        p=2
    ),

    MethodSpec(
        name="poly3",
        basis="poly",
        p=3
    ),

    MethodSpec(
        name="poly4",
        basis="poly",
        p=4
    ),

    MethodSpec(
        name="poly5",
        basis="poly",
        p=5
    ),

    # spline1 is identical to the linear model,
    # so it is omitted.

    MethodSpec(
        name="spline2",
        basis="spline",
        p=2
    ),

    MethodSpec(
        name="spline3",
        basis="spline",
        p=3
    ),

    MethodSpec(
        name="spline4",
        basis="spline",
        p=4
    ),

    MethodSpec(
        name="spline5",
        basis="spline",
        p=5
    ),
)


def select_methods(
    methods: Sequence[str | MethodSpec] | str | MethodSpec | None = None,
) -> tuple[MethodSpec, ...]:
    """Resolve user-friendly method names to validated ``MethodSpec`` objects.

    Examples
    --------
    ``select_methods(["linear", "poly3", "spline3"])``
    ``select_methods("spline4")``
    ``select_methods(None)``  # all defaults
    """

    if methods is None:
        return DEFAULT_METHODS
    if isinstance(methods, (str, MethodSpec)):
        methods = [methods]

    lookup = {method.name.lower(): method for method in DEFAULT_METHODS}
    resolved: list[MethodSpec] = []
    seen: set[str] = set()

    for requested in methods:
        if isinstance(requested, MethodSpec):
            method = requested
        else:
            name = str(requested).strip().lower()
            if name not in lookup:
                raise ValueError(
                    f"Unknown method {requested!r}. Available methods are "
                    f"{sorted(lookup)}."
                )
            method = lookup[name]

        if method.name not in seen:
            resolved.append(method)
            seen.add(method.name)

    if not resolved:
        raise ValueError("At least one temporal method must be supplied.")

    return tuple(resolved)


@dataclass
class SVTT_Scan:

    coord_path: str | Path = field(
        default_factory=lambda: DEFAULT_COORD_PATH
    )

    T_start: int = 1980
    T_end: int = 2000

    Bmc: int = 199

    # A flat list denotes one true cluster. A nested list may be used
    # when metrics should be evaluated against several true clusters.
    true_cluster: list = field(
        default_factory=lambda: [15, 29, 39]
    )

    # Backward-compatible reference values. When population_data is None,
    # Pop is used as a constant population. If population_data varies, the
    # default baseline rate remains E / Pop.
    Pop: float = 10000.0
    E: float = 100.0

    # Flexible population input used only by generate():
    #   scalar; county mapping; county vector; county-by-year matrix;
    #   DataFrame(county, year, population); or callable.
    population_data: Any = None

    # Optional scalar/year-specific baseline rate. When None, E / Pop is used.
    baseline_rate: Any = None

    n_sim: int = 1
    alpha: float = 0.05

    min_size: int = 2
    max_size_percent: float = 25.0
    step: int = 1

    seed: int = 218720

    # Maximum number of non-overlapping clusters to select.
    # Pass either nclust_max=K or n_max=K.
    nclust_max: int = 1
    n_max: int | None = None

    # Accepted scenario numbers
    par: tuple = (1, 2, 3)
    inc: tuple = (1, 2, 3)
    dec: tuple = (1, 2, 3)
    cyc: tuple = (1, 2, 3, 4)
    linear: tuple = (1, 2, 3, 4)
    non_linear: tuple = (1, 2, 3)

    def __post_init__(self):

        # =====================================================
        # Validate settings
        # =====================================================
        if self.T_start >= self.T_end:
            raise ValueError(
                "T_start must be smaller than T_end so that "
                "at least two time points are available."
            )

        if self.n_sim < 1:
            raise ValueError(
                "n_sim must be at least 1."
            )

        if self.Bmc < 1:
            raise ValueError(
                "Bmc must be at least 1."
            )

        if self.E <= 0:
            raise ValueError(
                "E must be positive."
            )

        if self.Pop <= 0:
            raise ValueError(
                "Pop must be positive."
            )

        if not 0 < self.alpha < 1:
            raise ValueError(
                "alpha must be between 0 and 1."
            )

        if self.min_size < 1:
            raise ValueError(
                "min_size must be at least 1."
            )

        if not 0 < self.max_size_percent <= 100:
            raise ValueError(
                "max_size_percent must be in (0, 100]."
            )

        if self.step < 1:
            raise ValueError(
                "step must be at least 1."
            )

        if self.n_max is not None:
            if not isinstance(self.n_max, (int, np.integer)):
                raise TypeError("n_max must be an integer.")
            self.nclust_max = int(self.n_max)

        if not isinstance(self.nclust_max, (int, np.integer)):
            raise TypeError("nclust_max must be an integer.")

        self.nclust_max = int(self.nclust_max)
        self.n_max = self.nclust_max

        if self.nclust_max < 1:
            raise ValueError(
                "nclust_max/n_max must be at least 1."
            )

        # =====================================================
        # Read coordinates
        # =====================================================
        coord_path = Path(self.coord_path).expanduser()

        if not coord_path.is_absolute():
            coord_path = SVTT_ROOT / coord_path

        coord_path = coord_path.resolve()

        if not coord_path.is_file():
            raise FileNotFoundError(
                "The county-coordinate file was not found. Expected: "
                f"{coord_path}. Keep nm_county_coordinates.txt in the "
                "downloaded SVTT folder, pass coord_path explicitly, or "
                "set the SVTT_ROOT environment variable."
            )

        self.coord_path = coord_path

        self.coords = pd.read_csv(
            self.coord_path,
            sep=r"\s+",
            header=None,
            names=["county", "x", "y"]
        )

        self.coords["county"] = (
            self.coords["county"].astype(int)
        )

        if self.coords["county"].duplicated().any():
            duplicates = (
                self.coords.loc[
                    self.coords["county"].duplicated(),
                    "county"
                ]
                .unique()
                .tolist()
            )

            raise ValueError(
                "Duplicate counties found in coordinate file: "
                f"{duplicates}"
            )

        if self.coords[
            ["county", "x", "y"]
        ].isna().any().any():
            raise ValueError(
                "Coordinate data contain missing values."
            )

        # Keep coordinates in county order
        self.coords = (
            self.coords
            .sort_values("county")
            .reset_index(drop=True)
        )

        self.counties = (
            self.coords["county"]
            .to_numpy(dtype=int)
        )

        # =====================================================
        # Time arrays
        # =====================================================
        self.years = np.arange(
            self.T_start,
            self.T_end + 1,
            dtype=int
        )

        self.t = np.arange(
            1,
            len(self.years) + 1,
            dtype=float
        )

        # =====================================================
        # True cluster(s)
        # =====================================================
        raw_true = self.true_cluster

        if raw_true is None:
            self.true_clusters = []
        elif (
            isinstance(raw_true, (list, tuple, set, np.ndarray, pd.Series))
            and len(raw_true) > 0
            and isinstance(
                list(raw_true)[0],
                (list, tuple, set, np.ndarray, pd.Series),
            )
        ):
            self.true_clusters = [
                np.atleast_1d(cluster).astype(int).tolist()
                for cluster in raw_true
            ]
        else:
            self.true_clusters = [
                np.atleast_1d(raw_true).astype(int).tolist()
            ]

        # Flatten for backward compatibility and for binary recovery metrics.
        self.true_cluster = []
        seen_true = set()
        for cluster in self.true_clusters:
            for county in cluster:
                county = int(county)
                if county not in seen_true:
                    seen_true.add(county)
                    self.true_cluster.append(county)

        missing = set(self.true_cluster) - set(self.counties)
        if missing:
            raise ValueError(
                "True-cluster counties were not found in "
                f"the coordinate file: {sorted(missing)}"
            )

        self._true_cluster_set = set(self.true_cluster)
        self._true_cluster_mask = np.isin(
            self.counties,
            self.true_cluster,
        )

        # =====================================================
        # Cached objects
        # =====================================================
        self._basis_cache = {}
        self._design_cache = {}
        self._null_stats_cache = {}

        # Created later by create_candidate_clusters()
        self.candidate_clusters = []

        self.candidate_masks = np.empty(
            (0, len(self.counties)),
            dtype=float
        )

        # Populated by create_candidate_clusters().  These objects make the
        # population-window rule explicit and auditable in saved results.
        self.candidate_population_reference = np.array([], dtype=float)
        self.candidate_population = np.array([], dtype=float)
        self.candidate_population_percent = np.array([], dtype=float)
        self.population_reference_total = np.nan

    def _resolve_population_matrix(self):
        """Return a positive county-by-year population matrix for generation."""

        n_counties = len(self.counties)
        n_times = len(self.years)
        source = self.Pop if self.population_data is None else self.population_data

        if callable(source):
            values = source(self.counties.copy(), self.years.copy())
            population_matrix = np.asarray(values, dtype=float)

        elif isinstance(source, pd.DataFrame):
            required = {"county", "year", "population"}
            missing = required - set(source.columns)
            if missing:
                raise ValueError(
                    "population_data DataFrame is missing columns: "
                    f"{sorted(missing)}"
                )

            population_df = source.loc[:, ["county", "year", "population"]].copy()
            population_df["county"] = pd.to_numeric(
                population_df["county"], errors="raise"
            ).astype(int)
            population_df["year"] = pd.to_numeric(
                population_df["year"], errors="raise"
            ).astype(int)
            population_df["population"] = pd.to_numeric(
                population_df["population"], errors="raise"
            ).astype(float)

            if population_df.duplicated(["county", "year"]).any():
                raise ValueError(
                    "population_data contains duplicated county-year rows."
                )

            pivot = population_df.pivot(
                index="county",
                columns="year",
                values="population",
            ).reindex(index=self.counties, columns=self.years)

            if pivot.isna().any().any():
                raise ValueError(
                    "population_data does not contain every required "
                    "county-year combination."
                )

            population_matrix = pivot.to_numpy(dtype=float)

        elif isinstance(source, Mapping):
            rows = []
            for county in self.counties:
                if int(county) not in source:
                    raise ValueError(
                        f"No population was supplied for county {int(county)}."
                    )
                county_values = np.asarray(source[int(county)], dtype=float)
                if county_values.ndim == 0:
                    county_values = np.full(n_times, float(county_values))
                else:
                    county_values = county_values.reshape(-1)
                    if county_values.size != n_times:
                        raise ValueError(
                            f"Population for county {int(county)} has "
                            f"{county_values.size} values; expected {n_times}."
                        )
                rows.append(county_values)
            population_matrix = np.vstack(rows)

        elif np.isscalar(source):
            population_matrix = np.full(
                (n_counties, n_times),
                float(source),
                dtype=float,
            )

        else:
            values = np.asarray(source, dtype=float)
            if values.shape == (n_counties,):
                population_matrix = np.repeat(values[:, None], n_times, axis=1)
            elif values.shape == (n_counties, n_times):
                population_matrix = values.copy()
            elif values.size == n_counties * n_times:
                population_matrix = values.reshape(n_counties, n_times)
            else:
                raise ValueError(
                    "population_data must be a scalar, mapping, callable, "
                    "DataFrame, county vector, or county-by-year matrix."
                )

        if population_matrix.shape != (n_counties, n_times):
            raise ValueError(
                f"Population matrix has shape {population_matrix.shape}; "
                f"expected {(n_counties, n_times)}."
            )

        if np.any(~np.isfinite(population_matrix)):
            raise ValueError("Population contains non-finite values.")
        if np.any(population_matrix <= 0):
            raise ValueError("Population values must be positive.")

        return population_matrix

    def _resolve_baseline_rate(self):
        """Return the baseline disease rate for every study year."""

        if self.baseline_rate is None:
            if not np.isscalar(self.Pop) or float(self.Pop) <= 0:
                raise ValueError(
                    "Pop must be a positive scalar when baseline_rate is None."
                )
            rates = np.full(
                len(self.years),
                float(self.E) / float(self.Pop),
                dtype=float,
            )
        else:
            source = self.baseline_rate
            if callable(source):
                source = source(self.years.copy())

            if isinstance(source, pd.Series):
                if set(self.years).issubset(set(source.index)):
                    rates = source.reindex(self.years).to_numpy(dtype=float)
                else:
                    rates = source.to_numpy(dtype=float)
            elif np.isscalar(source):
                rates = np.full(len(self.years), float(source), dtype=float)
            else:
                rates = np.asarray(source, dtype=float).reshape(-1)

            if rates.size != len(self.years):
                raise ValueError(
                    f"baseline_rate has {rates.size} values; "
                    f"expected {len(self.years)}."
                )

        if np.any(~np.isfinite(rates)) or np.any(rates <= 0):
            raise ValueError("Baseline rates must be finite and positive.")

        return rates

    @staticmethod
    def _flatten_clusters(clusters):
        """Recursively flatten one cluster or a collection of clusters."""

        if clusters is None:
            return set()

        if isinstance(clusters, float) and np.isnan(clusters):
            return set()

        if isinstance(clusters, (list, tuple, set, np.ndarray, pd.Series)):
            result = set()
            for item in clusters:
                if isinstance(item, (list, tuple, set, np.ndarray, pd.Series)):
                    result.update(SVTT_Scan._flatten_clusters(item))
                elif item is not None and not (
                    isinstance(item, float) and np.isnan(item)
                ):
                    result.add(int(item))
            return result

        return {int(clusters)}

    def get_scenario_functions(
        self,
        scenario_name,
        scenario_number
    ):
        """
        Return the inside- and outside-cluster relative-risk
        functions for a requested scenario.
        """

        scenario_name = str(
            scenario_name
        ).strip().lower()

        scenario_map = {
            "linear": (
                self.linear,
                self._linear_scenario
            ),
            "lin": (
                self.linear,
                self._linear_scenario
            ),
            "parabolic": (
                self.par,
                self._parabolic_scenario
            ),
            "par": (
                self.par,
                self._parabolic_scenario
            ),
            "increasing": (
                self.inc,
                self._increasing_scenario
            ),
            "inc": (
                self.inc,
                self._increasing_scenario
            ),
            "decreasing": (
                self.dec,
                self._decreasing_scenario
            ),
            "dec": (
                self.dec,
                self._decreasing_scenario
            ),
            "cyclic": (
                self.cyc,
                self._cyclic_scenario
            ),
            "cyc": (
                self.cyc,
                self._cyclic_scenario
            ),
            "residual": (
                self.non_linear,
                self._nonlinear_residual_scenario
            ),
            "res": (
                self.non_linear,
                self._nonlinear_residual_scenario
            ),
            "nonlinear": (
                self.non_linear,
                self._nonlinear_residual_scenario
            ),
            "non-linear": (
                self.non_linear,
                self._nonlinear_residual_scenario
            ),
            "nonlinear_residual": (
                self.non_linear,
                self._nonlinear_residual_scenario
            )
        }

        if scenario_name not in scenario_map:
            raise ValueError(
                "scenario_name must be one of: "
                "linear, parabolic, increasing, decreasing, "
                "cyclic, or residual."
            )

        valid_numbers, scenario_function = (
            scenario_map[scenario_name]
        )

        if scenario_number not in valid_numbers:
            raise ValueError(
                f"{scenario_name} scenario must be one of "
                f"{valid_numbers}."
            )

        return scenario_function(
            scenario_number
        )

    def generate(
        self,
        scenario_name,
        scenario_number
    ):
        """
        Generate spatial-temporal Poisson datasets.

        Population may be constant or vary by county and year. Expected
        baseline counts are population multiplied by the baseline rate.
        """

        inside_fun, outside_fun = self.get_scenario_functions(
            scenario_name,
            scenario_number,
        )

        theta_inside = np.asarray(inside_fun(self.t), dtype=float)
        theta_outside = np.asarray(outside_fun(self.t), dtype=float)

        if theta_inside.shape != self.t.shape:
            raise ValueError(
                f"Inside theta has shape {theta_inside.shape}; "
                f"expected {self.t.shape}."
            )
        if theta_outside.shape != self.t.shape:
            raise ValueError(
                f"Outside theta has shape {theta_outside.shape}; "
                f"expected {self.t.shape}."
            )
        if (
            np.any(~np.isfinite(theta_inside))
            or np.any(~np.isfinite(theta_outside))
            or np.any(theta_inside <= 0)
            or np.any(theta_outside <= 0)
        ):
            raise ValueError(
                "Relative-risk curves must be finite and positive."
            )

        n_counties = len(self.counties)
        n_times = len(self.years)

        population_matrix = self._resolve_population_matrix()
        baseline_rates = self._resolve_baseline_rate()
        E_matrix = population_matrix * baseline_rates[None, :]

        theta_matrix = np.where(
            self._true_cluster_mask[:, None],
            theta_inside[None, :],
            theta_outside[None, :],
        )
        lambda_matrix = E_matrix * theta_matrix

        county_column = np.repeat(self.counties, n_times)
        year_column = np.tile(self.years, n_counties)
        cluster_column = np.repeat(self._true_cluster_mask, n_times)
        x_column = np.repeat(
            self.coords["x"].to_numpy(dtype=float), n_times
        )
        y_column = np.repeat(
            self.coords["y"].to_numpy(dtype=float), n_times
        )
        population_column = population_matrix.reshape(-1)
        E_column = E_matrix.reshape(-1)

        simulation_frames = []
        for sim in range(1, self.n_sim + 1):
            rng = np.random.default_rng(
                np.random.SeedSequence([int(self.seed), int(sim)])
            )
            observed_cases = rng.poisson(lam=lambda_matrix).reshape(-1)

            simulation_frames.append(pd.DataFrame({
                "sim": sim,
                "county": county_column,
                "x": x_column,
                "y": y_column,
                "cluster": cluster_column,
                "population": population_column,
                "E": E_column,
                "observed_cases": observed_cases,
                "year": year_column,
            }))

        return pd.concat(simulation_frames, ignore_index=True)

    def _baseline_curve(self, t):
        """Baseline relative risk outside the true cluster."""
        return np.ones_like(
            np.asarray(t, dtype=float)
        )

    def _scale_time(self, t):
        """Scale time to [0, 1] using the complete study period."""
        t = np.asarray(t, dtype=float)

        denominator = (
            self.t.max() - self.t.min()
        )

        if denominator <= 0:
            raise ValueError(
                "At least two time points are required."
            )

        return (
            (t - self.t.min())
            / denominator
        )

    def _nonlinear_residual_scenario(
        self,
        scenario_number
    ):
        """
        Generate a nonlinear signal after removing its fitted
        intercept, linear, and quadratic components.

        Scenarios
        ---------
        1 : S-shaped signal
        2 : Two-bend signal
        3 : Narrow-bump signal
        """

        if scenario_number not in self.non_linear:
            raise ValueError(
                "Nonlinear residual scenario must be one of "
                f"{self.non_linear}."
            )

        t_reference = np.asarray(
            self.t,
            dtype=float
        )

        t_min = t_reference.min()
        t_max = t_reference.max()
        t_range = t_max - t_min
        centre = (t_min + t_max) / 2

        if t_range <= 0:
            raise ValueError(
                "At least two time points are required."
            )

        if scenario_number == 1:

            steepness = 0.55

            logistic_min = 1.0 / (
                1.0
                + np.exp(
                    -steepness
                    * (t_min - centre)
                )
            )

            logistic_max = 1.0 / (
                1.0
                + np.exp(
                    -steepness
                    * (t_max - centre)
                )
            )

            logistic_range = (
                logistic_max - logistic_min
            )

            def original_curve(t):
                t = np.asarray(t, dtype=float)

                y = 1.0 / (
                    1.0
                    + np.exp(
                        -steepness
                        * (t - centre)
                    )
                )

                return (
                    (y - logistic_min)
                    / logistic_range
                )

        elif scenario_number == 2:

            def original_curve(t):
                t = np.asarray(t, dtype=float)

                scaled = (
                    2.0
                    * np.pi
                    * (t - t_min)
                    / t_range
                )

                return np.sin(scaled)

        else:

            width = 0.06 * t_range

            def original_curve(t):
                t = np.asarray(t, dtype=float)

                return np.exp(
                    -0.5
                    * ((t - centre) / width) ** 2
                )

        # Fit the quadratic model only once
        y_original = original_curve(
            t_reference
        )

        X_reference = np.column_stack([
            np.ones_like(t_reference),
            t_reference,
            t_reference**2
        ])

        beta_hat, *_ = np.linalg.lstsq(
            X_reference,
            y_original,
            rcond=None
        )

        residual_reference = (
            y_original
            - X_reference @ beta_hat
        )

        residual_scale = np.max(
            np.abs(residual_reference)
        )

        if np.isclose(residual_scale, 0.0):
            raise ValueError(
                "The nonlinear residual is identically zero."
            )

        effect_size = 0.5

        def inside_fun(t):
            t = np.asarray(t, dtype=float)

            X_new = np.column_stack([
                np.ones_like(t),
                t,
                t**2
            ])

            residual = (
                original_curve(t)
                - X_new @ beta_hat
            )

            standardized_residual = (
                residual / residual_scale
            )

            theta = (
                1.0
                + effect_size
                * standardized_residual
            )

            if np.any(~np.isfinite(theta)):
                raise ValueError(
                    "The nonlinear relative-risk curve contains "
                    "non-finite values."
                )

            if np.any(theta <= 0):
                raise ValueError(
                    "The nonlinear relative-risk curve contains "
                    "non-positive values."
                )

            return theta

        return inside_fun, self._baseline_curve

    def _linear_scenario(self, scenario):

        if scenario not in self.linear:
            raise ValueError(
                f"Linear scenario must be one of {self.linear}."
            )

        endpoints = (
            (3.0, 0.1),
            (2.0, 1.0),
            (0.1, 3.0),
            (1.0, 2.0)
        )

        start_rr, end_rr = endpoints[
            scenario - 1
        ]

        def inside_fun(t):
            u = self._scale_time(t)

            return (
                start_rr
                + (end_rr - start_rr) * u
            )

        return inside_fun, self._baseline_curve

    def _parabolic_scenario(self, scenario):

        if scenario not in self.par:
            raise ValueError(
                f"Parabolic scenario must be one of {self.par}."
            )

        beta2_values = (
            0.015,
            -0.015,
            -0.03
        )

        R_values = (
            1.5,
            2.0,
            3.0
        )

        beta2 = beta2_values[
            scenario - 1
        ]

        R = R_values[
            scenario - 1
        ]

        reference_curve = (
            beta2
            * (self.t - 4.0)
            * (self.t - 19.0)
        )

        maximum = np.max(
            reference_curve
        )

        if np.isclose(maximum, 0.0):
            raise ValueError(
                "Cannot scale the parabolic curve because its "
                "maximum is zero."
            )

        scale_factor = (
            np.log(R) / maximum
        )

        def inside_fun(t):
            t = np.asarray(t, dtype=float)

            curve = (
                beta2
                * (t - 4.0)
                * (t - 19.0)
            )

            return np.exp(
                scale_factor * curve
            )

        return inside_fun, self._baseline_curve

    def _method_cache_key(
        self,
        method: MethodSpec
        ):
        return (
            method.basis,
            int(method.p),
        )

    def _increasing_scenario(self, scenario):

        if scenario not in self.inc:
            raise ValueError(
                f"Increasing scenario must be one of {self.inc}."
            )

        lambda_values = (
            5.5,
            2.0,
            -2.0
        )

        gamma_values = (
            0.85,
            1.00,
            1.15
        )

        lam = lambda_values[
            scenario - 1
        ]

        gamma = gamma_values[
            scenario - 1
        ]

        log_RR_start = np.log(0.3)
        log_RR_end = np.log(3.0)

        denominator = (
            1.0 - np.exp(-lam)
        )

        def inside_fun(t):
            u = self._scale_time(t)

            if np.isclose(lam, 0.0):
                H = u
            else:
                H = (
                    1.0 - np.exp(-lam * u)
                ) / denominator

            G = H**gamma

            return np.exp(
                log_RR_start
                + (
                    log_RR_end
                    - log_RR_start
                ) * G
            )

        return inside_fun, self._baseline_curve

    def _decreasing_scenario(self, scenario):

        if scenario not in self.dec:
            raise ValueError(
                f"Decreasing scenario must be one of {self.dec}."
            )

        lambda_values = (
            2.0,
            -2.0,
            -6.5
        )

        gamma_values = (
            1.00,
            1.15,
            1.38
        )

        lam = lambda_values[
            scenario - 1
        ]

        gamma = gamma_values[
            scenario - 1
        ]

        log_RR_start = np.log(3.0)
        log_RR_end = np.log(0.3)

        denominator = (
            1.0 - np.exp(-lam)
        )

        def inside_fun(t):
            u = self._scale_time(t)

            if np.isclose(lam, 0.0):
                H = u
            else:
                H = (
                    1.0 - np.exp(-lam * u)
                ) / denominator

            G = H**gamma

            return np.exp(
                log_RR_start
                + (
                    log_RR_end
                    - log_RR_start
                ) * G
            )

        return inside_fun, self._baseline_curve

    def _cyclic_scenario(self, scenario):

        if scenario not in self.cyc:
            raise ValueError(
                f"Cyclic scenario must be one of {self.cyc}."
            )

        amplitude_values = (
            1.2,
            1.5,
            2.0,
            3.0
        )

        frequency_values = (
            1,
            2,
            3,
            4
        )

        amplitude = amplitude_values[
            scenario - 1
        ]

        frequency = frequency_values[
            scenario - 1
        ]

        RR_min = 0.01

        # Use the complete time range to obtain fixed scaling
        reference_u = self._scale_time(
            self.t
        )

        reference_raw = np.sin(
            2.0
            * np.pi
            * frequency
            * reference_u
        )

        raw_min = reference_raw.min()
        raw_range = (
            reference_raw.max() - raw_min
        )

        if np.isclose(raw_range, 0.0):
            raise ValueError(
                "The cyclic curve has zero variation."
            )

        def inside_fun(t):
            u = self._scale_time(t)

            raw = np.sin(
                2.0
                * np.pi
                * frequency
                * u
            )

            standardized = (
                (raw - raw_min)
                / raw_range
            )

            return (
                RR_min
                + (amplitude - RR_min)
                * standardized
            )

        return inside_fun, self._baseline_curve

    def _resolve_candidate_population(self, population_data=None):
        """Return one population-at-risk value per location.

        Candidate windows are spatial and remain fixed over time.  Therefore,
        when county-by-time populations are supplied, each county's mean
        population over the study period is used.  Equivalently, because all
        counties have the same number of time points, this is the county's
        share of total person-time.  For the simulation design in which the
        2021 population is held constant over all 21 times, this reduces
        exactly to the supplied 2021 population ``P_i``.
        """

        source = population_data
        if source is None:
            source = (
                self.population_data
                if self.population_data is not None
                else self.Pop
            )

        n_counties = len(self.counties)

        if isinstance(source, pd.DataFrame):
            required = {"county", "population"}
            missing = required - set(source.columns)
            if missing:
                raise ValueError(
                    "Population data for candidate windows are missing "
                    f"columns: {sorted(missing)}"
                )

            frame = source.loc[:, ["county", "population"]].copy()
            frame["county"] = pd.to_numeric(
                frame["county"], errors="raise"
            ).astype(int)
            frame["population"] = pd.to_numeric(
                frame["population"], errors="raise"
            ).astype(float)

            extra = sorted(set(frame["county"]) - set(self.counties))
            if extra:
                raise ValueError(
                    "Population data contain locations absent from the "
                    f"coordinate file: {extra}"
                )

            values = (
                frame.groupby("county", sort=False)["population"]
                .mean()
                .reindex(self.counties)
            )
            if values.isna().any():
                missing_counties = self.counties[
                    values.isna().to_numpy()
                ].tolist()
                raise ValueError(
                    "No population was supplied for locations: "
                    f"{missing_counties}"
                )
            population = values.to_numpy(dtype=float)

        elif isinstance(source, pd.Series):
            if set(self.counties).issubset(set(source.index)):
                population = source.reindex(self.counties).to_numpy(dtype=float)
            elif len(source) == n_counties:
                population = source.to_numpy(dtype=float)
            else:
                raise ValueError(
                    "Population Series must be indexed by location ID or have "
                    f"length {n_counties}."
                )

        elif isinstance(source, Mapping):
            rows = []
            for county in self.counties:
                if int(county) not in source:
                    raise ValueError(
                        f"No population was supplied for location {int(county)}."
                    )
                county_values = np.asarray(source[int(county)], dtype=float)
                rows.append(float(np.mean(county_values)))
            population = np.asarray(rows, dtype=float)

        elif np.isscalar(source):
            population = np.full(n_counties, float(source), dtype=float)

        else:
            values = np.asarray(source, dtype=float)
            if values.shape == (n_counties,):
                population = values.copy()
            elif values.ndim == 2 and values.shape[0] == n_counties:
                population = values.mean(axis=1)
            elif values.size % n_counties == 0:
                population = values.reshape(n_counties, -1).mean(axis=1)
            else:
                raise ValueError(
                    "Candidate-window population must be a scalar, mapping, "
                    "DataFrame, location vector, or location-by-time matrix."
                )

        population = np.asarray(population, dtype=float).reshape(-1)
        if population.shape != (n_counties,):
            raise ValueError(
                f"Candidate population has shape {population.shape}; "
                f"expected {(n_counties,)}."
            )
        if np.any(~np.isfinite(population)):
            raise ValueError("Candidate-window population is non-finite.")
        if np.any(population <= 0):
            raise ValueError("Candidate-window population must be positive.")

        return population

    def create_candidate_clusters(self, population_data=None):
        """Create circular candidates limited by population percentage.

        A nearest-neighbour prefix ``Z`` is retained only when

            100 * sum(i in Z) P_i / sum(i) P_i <= max_size_percent.

        Thus ``max_size_percent=25`` means at most 25% of the population at
        risk, not 25% of the number of locations.  With equal populations the
        two definitions coincide.
        """

        number_of_counties = len(self.counties)
        population = self._resolve_candidate_population(population_data)
        total_population = float(population.sum())
        maximum_population = (
            total_population * float(self.max_size_percent) / 100.0
        )
        numerical_tolerance = max(1e-9, 1e-12 * total_population)

        coordinates = self.coords[["x", "y"]].to_numpy(dtype=float)
        distance_matrix = cdist(coordinates, coordinates)
        nearest_orders = np.argsort(distance_matrix, axis=1)

        candidate_clusters: list[list[int]] = []
        candidate_indices: list[np.ndarray] = []
        candidate_population: list[float] = []
        candidate_population_percent: list[float] = []
        seen: set[tuple[int, ...]] = set()

        for centre_index in range(number_of_counties):
            nearest = nearest_orders[centre_index]
            cumulative_population = 0.0

            # The final all-location window is never a valid scan candidate.
            for size in range(1, number_of_counties):
                cumulative_population += float(population[nearest[size - 1]])

                # Population is positive, so every larger prefix will also
                # exceed the limit once this condition is reached.
                if cumulative_population > maximum_population + numerical_tolerance:
                    break

                if size < self.min_size:
                    continue
                if (size - self.min_size) % self.step != 0:
                    continue

                indices = np.asarray(nearest[:size], dtype=int)
                unique_key = tuple(sorted(indices.tolist()))
                if unique_key in seen:
                    continue
                seen.add(unique_key)

                cluster_population = float(population[indices].sum())
                cluster_percent = 100.0 * cluster_population / total_population

                candidate_indices.append(indices)
                candidate_clusters.append(self.counties[indices].tolist())
                candidate_population.append(cluster_population)
                candidate_population_percent.append(cluster_percent)

        if not candidate_clusters:
            smallest_share = 100.0 * float(population.min()) / total_population
            raise ValueError(
                "No candidate window satisfies the requested settings. "
                f"min_size={self.min_size}, max_size_percent="
                f"{self.max_size_percent}, and the smallest one-location "
                f"population share is {smallest_share:.6f}%."
            )

        candidate_masks = np.zeros(
            (len(candidate_clusters), number_of_counties),
            dtype=float,
        )
        for candidate_id, indices in enumerate(candidate_indices):
            candidate_masks[candidate_id, indices] = 1.0

        population_percent_array = np.asarray(
            candidate_population_percent, dtype=float
        )
        if np.any(
            population_percent_array
            > float(self.max_size_percent) + 1e-10
        ):
            raise RuntimeError(
                "Internal error: a candidate exceeds max_size_percent."
            )

        self.candidate_clusters = candidate_clusters
        self.candidate_masks = candidate_masks
        self.candidate_population_reference = population.copy()
        self.candidate_population = np.asarray(
            candidate_population, dtype=float
        )
        self.candidate_population_percent = population_percent_array
        self.population_reference_total = total_population

        # Candidate configuration changed, so cached null results are invalid.
        self._null_stats_cache.clear()
        return candidate_clusters

    @staticmethod
    def _positive_cube(x):

        x = np.asarray(
            x,
            dtype=float
        )

        return np.maximum(
            x,
            0.0
        ) ** 3

    @staticmethod
    def _center_and_orthogonalize(
        basis,
        p,
        tol=1e-10
    ):

        basis = np.asarray(
            basis,
            dtype=float
        )

        if basis.ndim != 2:
            raise ValueError(
                "The raw basis must be two-dimensional."
            )

        if basis.shape[1] != p:
            raise ValueError(
                f"The raw basis has {basis.shape[1]} columns; "
                f"expected p={p}."
            )

        centred_basis = (
            basis
            - basis.mean(
                axis=0,
                keepdims=True
            )
        )

        rank = np.linalg.matrix_rank(
            centred_basis,
            tol=tol
        )

        if rank < p:
            raise ValueError(
                f"Basis rank {rank} is smaller than p={p}."
            )

        Q, _ = np.linalg.qr(
            centred_basis,
            mode="reduced"
        )

        return np.asarray(
            Q[:, :p],
            dtype=float
        )

    def _polynomial_basis(
        self,
        x,
        p
    ):

        x = np.asarray(
            x,
            dtype=float
        )

        # Transform [0, 1] to [-1, 1]
        z = 2.0 * x - 1.0

        raw_basis = np.column_stack([
            z**power
            for power in range(
                1,
                p + 1
            )
        ])

        return self._center_and_orthogonalize(
            raw_basis,
            p
        )

    def _spline_basis(
        self,
        x,
        p
    ):
        """
        Restricted cubic spline with p basis columns
        and p-1 internal knots.
        """

        x = np.asarray(
            x,
            dtype=float
        )

        # spline1 is equivalent to linear
        if p == 1:
            return self._polynomial_basis(
                x,
                p=1
            )

        # Produces p-1 internal-knot probabilities
        probabilities = np.linspace(
            0.0,
            1.0,
            p + 1
        )[1:-1]

        internal_knots = np.asarray(
            np.quantile(
                x,
                probabilities
            ),
            dtype=float
        )

        internal_knots = np.atleast_1d(
            internal_knots
        )

        if np.unique(internal_knots).size != p - 1:
            raise ValueError(
                f"Could not construct {p - 1} unique "
                f"internal knots for spline{p}."
            )

        knots = np.concatenate([
            [x.min()],
            internal_knots,
            [x.max()]
        ])

        right = knots[-1]
        next_right = knots[-2]
        denominator = right - next_right

        if denominator <= 0:
            raise ValueError(
                "Invalid restricted cubic spline knots."
            )

        columns = [x]

        for knot in knots[:-2]:

            column = (
                self._positive_cube(
                    x - knot
                )
                - (
                    (right - knot)
                    / denominator
                )
                * self._positive_cube(
                    x - next_right
                )
                + (
                    (next_right - knot)
                    / denominator
                )
                * self._positive_cube(
                    x - right
                )
            )

            columns.append(
                column
            )

        raw_basis = np.column_stack(
            columns
        )

        return self._center_and_orthogonalize(
            raw_basis,
            p
        )

    def _make_basis(
        self,
        method: MethodSpec
    ):

        key = self._method_cache_key(
            method
        )

        cached = self._basis_cache.get(
            key
        )

        if cached is not None:
            return cached

        scaled_time = np.asarray(
            self._scale_time(self.t),
            dtype=float
        )

        number_of_times = scaled_time.size
        p = int(method.p)

        if number_of_times < 2:
            raise ValueError(
                "At least two time points are required."
            )

        if not np.all(np.isfinite(scaled_time)):
            raise ValueError(
                "Scaled time contains non-finite values."
            )

        if np.unique(scaled_time).size != number_of_times:
            raise ValueError(
                "Time points must be distinct."
            )

        # p basis coefficients plus one intercept
        number_of_parameters = p + 1

        if number_of_parameters >= number_of_times:
            raise ValueError(
                f"{method.name!r} has too many parameters for "
                f"{number_of_times} time points. "
                f"Basis columns: {p}; "
                f"parameters including intercept: "
                f"{number_of_parameters}."
            )

        if method.basis == "poly":

            basis = self._polynomial_basis(
                scaled_time,
                p
            )

        elif method.basis == "spline":

            basis = self._spline_basis(
                scaled_time,
                p
            )

        else:
            raise ValueError(
                "method.basis must be either 'poly' or 'spline'."
            )

        expected_shape = (
            number_of_times,
            p
        )

        if basis.shape != expected_shape:
            raise ValueError(
                f"The basis for {method.name!r} has shape "
                f"{basis.shape}; expected {expected_shape}."
            )

        if not np.all(np.isfinite(basis)):
            raise ValueError(
                f"Non-finite basis values were generated "
                f"for {method.name!r}."
            )

        design_with_intercept = np.column_stack([
            np.ones(number_of_times),
            basis
        ])

        design_rank = np.linalg.matrix_rank(
            design_with_intercept
        )

        if design_rank < design_with_intercept.shape[1]:
            raise ValueError(
                f"The design matrix for {method.name!r} "
                "is rank deficient."
            )

        basis.setflags(
            write=False
        )

        self._basis_cache[key] = basis

        return basis

    def _get_design_matrices(
        self,
        method: MethodSpec
    ):
        """Create and cache alternative and null design matrices."""

        key = self._method_cache_key(method)

        cached = self._design_cache.get(key)

        if cached is not None:
            return cached

        basis = self._make_basis(
            method
        )

        number_of_times = len(
            self.t
        )

        # Design matrix used for separate inside/outside fits
        X_alt = np.column_stack([
            np.ones(number_of_times),
            basis
        ])

        # Null model:
        # separate inside/outside intercepts,
        # but a common temporal curve
        X_null = np.column_stack([
            np.concatenate([
                np.ones(number_of_times),
                np.zeros(number_of_times)
            ]),
            np.concatenate([
                np.zeros(number_of_times),
                np.ones(number_of_times)
            ]),
            np.vstack([
                basis,
                basis
            ])
        ])

        if (
            np.linalg.matrix_rank(X_alt)
            < X_alt.shape[1]
        ):
            raise ValueError(
                f"Alternative design matrix for "
                f"{method.name!r} is rank deficient."
            )

        if (
            np.linalg.matrix_rank(X_null)
            < X_null.shape[1]
        ):
            raise ValueError(
                f"Null design matrix for "
                f"{method.name!r} is rank deficient."
            )

        X_alt.setflags(write=False)
        X_null.setflags(write=False)

        self._design_cache[key] = (
            X_alt,
            X_null
        )

        return X_alt, X_null


    def _prepare_scan_arrays(
        self,
        df_sim
        ):
        
        """
        Convert one simulation to county-by-time matrices and
        calculate expected counts using a single pooled reference rate:

            R0 = sum_i,t Y_it / sum_i,t P_it
            E_it = P_it * R0

        The denominator is total person-time: repeated annual populations
        contribute once per observed year. All counties and years enter R0.
        The same R0 is used inside and outside every candidate window.
        """

        required_columns = {
            "county",
            "year",
            "observed_cases",
            "population",
        }

        missing_columns = (
            required_columns
            - set(df_sim.columns)
        )

        if missing_columns:
            raise ValueError(
                "Missing scan columns: "
                f"{sorted(missing_columns)}"
            )

        number_of_counties = len(
            self.counties
        )

        number_of_times = len(
            self.years
        )

        expected_rows = (
            number_of_counties
            * number_of_times
        )

        if len(df_sim) != expected_rows:
            raise ValueError(
                f"Expected {expected_rows} rows for one simulation, "
                f"but received {len(df_sim)}."
            )

        ordered = (
            df_sim
            .sort_values(
                ["county", "year"]
            )
            .reset_index(drop=True)
        )

        actual_counties = (
            ordered["county"]
            .to_numpy()
            .reshape(
                number_of_counties,
                number_of_times,
            )
        )

        actual_years = (
            ordered["year"]
            .to_numpy()
            .reshape(
                number_of_counties,
                number_of_times,
            )
        )

        if not np.array_equal(
            actual_counties[:, 0],
            self.counties,
        ):
            raise ValueError(
                "The simulation contains unexpected or "
                "missing counties."
            )

        if not np.array_equal(
            actual_years[0],
            self.years,
        ):
            raise ValueError(
                "The simulation contains unexpected or "
                "missing years."
            )

        if not np.all(
            actual_years
            == self.years[None, :]
        ):
            raise ValueError(
                "County-year combinations are incomplete "
                "or duplicated."
            )

        # --------------------------------------------------------
        # Observed cases: county × year
        # --------------------------------------------------------
        y_matrix = (
            ordered["observed_cases"]
            .to_numpy(dtype=float)
            .reshape(
                number_of_counties,
                number_of_times,
            )
        )

        # --------------------------------------------------------
        # Population: county × year
        # --------------------------------------------------------
        population_matrix = (
            ordered["population"]
            .to_numpy(dtype=float)
            .reshape(
                number_of_counties,
                number_of_times,
            )
        )

        if np.any(~np.isfinite(y_matrix)):
            raise ValueError(
                "Observed cases contain non-finite values."
            )

        if np.any(y_matrix < 0):
            raise ValueError(
                "Observed cases cannot be negative."
            )

        if np.any(~np.isfinite(population_matrix)):
            raise ValueError(
                "Population contains non-finite values."
            )

        if np.any(population_matrix <= 0):
            raise ValueError(
                "Population values must be positive."
            )

        if not np.allclose(y_matrix, np.rint(y_matrix), rtol=0.0, atol=1e-8):
            raise ValueError("Observed Poisson counts must be integer-valued.")

        total_cases = float(y_matrix.sum())
        total_person_time = float(population_matrix.sum())
        if not np.isfinite(total_cases) or total_cases <= 0:
            raise ValueError("At least one case is required for pooled R0.")
        if not np.isfinite(total_person_time) or total_person_time <= 0:
            raise ValueError("Total population person-time must be positive.")

        pooled_R0 = total_cases / total_person_time
        E_matrix = population_matrix * pooled_R0
        if np.any(~np.isfinite(E_matrix)) or np.any(E_matrix <= 0):
            raise ValueError("Pooled-R0 expected counts must be finite and positive.")

        # Reconcile over the complete study; individual yearly totals need
        # not agree, because the reference rate is fixed over time.
        if not np.isclose(E_matrix.sum(), total_cases, rtol=1e-10, atol=1e-8):
            raise RuntimeError("Pooled expected and observed study totals differ.")

        self.pooled_R0 = float(pooled_R0)
        self.total_person_time = total_person_time
        self.total_observed_cases = total_cases
        return y_matrix, E_matrix

    def standardize_saved_data(self, data):
        """Return normalized rows with the exact R0, expected and SIR used.

        Accepts saved CSV columns county/year/cases/pop as well as the
        normalized observed_cases/population names used by the scanner.
        """
        ordered = _normalise_saved_dataset_columns(data).sort_values(
            ["county", "year"]
        ).reset_index(drop=True)
        y_matrix, E_matrix = self._prepare_scan_arrays(ordered)
        ordered["pooled_R0"] = self.pooled_R0
        ordered["expected_cases"] = E_matrix.reshape(-1)
        ordered["SIR"] = (y_matrix / E_matrix).reshape(-1)
        return ordered


    def _aggregate_all_candidates(
        self,
        y_matrix,
        E_matrix
    ):
        """
        Aggregate cases and expected counts for all candidate
        clusters using matrix multiplication.
        """

        y_matrix = np.asarray(
            y_matrix,
            dtype=float
        )

        E_matrix = np.asarray(
            E_matrix,
            dtype=float
        )

        expected_shape = (
            len(self.counties),
            len(self.years)
        )

        if y_matrix.shape != expected_shape:
            raise ValueError(
                f"y_matrix has shape {y_matrix.shape}; "
                f"expected {expected_shape}."
            )

        if E_matrix.shape != expected_shape:
            raise ValueError(
                f"E_matrix has shape {E_matrix.shape}; "
                f"expected {expected_shape}."
            )

        if len(self.candidate_clusters) == 0:
            self.create_candidate_clusters()

        expected_mask_shape = (
            len(self.candidate_clusters),
            len(self.counties)
        )

        if (
            self.candidate_masks.shape
            != expected_mask_shape
        ):
            raise ValueError(
                "Candidate masks are missing or inconsistent. "
                "Run create_candidate_clusters() again."
            )

        y_inside = (
            self.candidate_masks @ y_matrix
        )

        E_inside = (
            self.candidate_masks @ E_matrix
        )

        y_total = y_matrix.sum(
            axis=0,
            keepdims=True
        )

        E_total = E_matrix.sum(
            axis=0,
            keepdims=True
        )

        y_outside = y_total - y_inside
        E_outside = E_total - E_inside

        return (
            y_inside,
            E_inside,
            y_outside,
            E_outside
        )

    def _fit_poisson_mle(
        self,
        y,
        E,
        X,
        max_iter=40,
        tol=1e-7
    ):
        """
        Fit a Poisson log-linear model with an offset using IRLS.

        The returned log-likelihood omits log(y!), which cancels
        exactly when calculating the likelihood-ratio statistic.
        """

        y = np.asarray(y, dtype=float)
        E = np.asarray(E, dtype=float)
        X = np.asarray(X, dtype=float)

        if y.ndim != 1 or E.ndim != 1:
            raise ValueError(
                "y and E must be one-dimensional."
            )

        if y.shape != E.shape:
            raise ValueError(
                "y and E must have the same shape."
            )

        if X.shape[0] != len(y):
            raise ValueError(
                "The number of rows in X must match y."
            )

        if np.any(y < 0):
            raise ValueError(
                "Poisson observations cannot be negative."
            )

        if np.any(E <= 0):
            raise ValueError(
                "Expected counts must be positive."
            )

        beta = np.zeros(
            X.shape[1],
            dtype=float
        )

        offset = np.log(E)
        converged = False

        for _ in range(max_iter):

            linear_predictor = (
                offset + X @ beta
            )

            clipped_predictor = np.clip(
                linear_predictor,
                -30.0,
                30.0
            )

            mu = np.exp(
                clipped_predictor
            )

            working_response = (
                clipped_predictor
                - offset
                + (y - mu) / np.maximum(mu, 1e-12)
            )

            XtW = X.T * mu
            hessian = XtW @ X
            rhs = XtW @ working_response

            try:
                beta_new = np.linalg.solve(
                    hessian,
                    rhs
                )
            except np.linalg.LinAlgError:
                beta_new = np.linalg.lstsq(
                    hessian,
                    rhs,
                    rcond=None
                )[0]

            if np.any(~np.isfinite(beta_new)):
                break

            difference = np.max(
                np.abs(beta_new - beta)
            )

            beta = beta_new

            if difference < tol:
                converged = True
                break

        final_predictor = np.clip(
            offset + X @ beta,
            -30.0,
            30.0
        )

        mu = np.exp(
            final_predictor
        )

        loglik = float(
            np.sum(
                y * final_predictor - mu
            )
        )

        return {
            "loglik": loglik,
            "converged": (
                converged
                and np.isfinite(loglik)
            )
        }

    def _llr_from_aggregates(
        self,
        y_in,
        E_in,
        y_out,
        E_out,
        method
    ):
        """Calculate the LLR for one candidate cluster."""

        X_alt, X_null = (
            self._get_design_matrices(method)
        )

        fit_inside = self._fit_poisson_mle(
            y=y_in,
            E=E_in,
            X=X_alt
        )

        fit_outside = self._fit_poisson_mle(
            y=y_out,
            E=E_out,
            X=X_alt
        )

        y_null = np.concatenate([
            y_in,
            y_out
        ])

        E_null = np.concatenate([
            E_in,
            E_out
        ])

        fit_null = self._fit_poisson_mle(
            y=y_null,
            E=E_null,
            X=X_null
        )

        converged = (
            fit_inside["converged"]
            and fit_outside["converged"]
            and fit_null["converged"]
        )

        if not converged:
            return {
                "LLR": np.nan,
                "ll_alt": np.nan,
                "ll_null": np.nan,
                "converged": False
            }

        ll_alt = (
            fit_inside["loglik"]
            + fit_outside["loglik"]
        )

        ll_null = fit_null["loglik"]

        llr = max(
            0.0,
            float(ll_alt - ll_null)
        )

        return {
            "LLR": llr,
            "ll_alt": float(ll_alt),
            "ll_null": float(ll_null),
            "converged": True
        }

    def _prepare_candidate_aggregates(
        self,
        y_matrix,
        E_matrix
    ):
        """
        Aggregate all candidates once so the same arrays can be
        reused during the scan.
        """

        return self._aggregate_all_candidates(
            y_matrix=y_matrix,
            E_matrix=E_matrix
        )

    def _find_max_llr_from_arrays(
        self,
        y_matrix,
        E_matrix,
        method
    ):
        """
        Return only the maximum LLR. Used for Monte Carlo null
        samples to avoid creating and sorting DataFrames.
        """

        (
            y_inside,
            E_inside,
            y_outside,
            E_outside
        ) = self._prepare_candidate_aggregates(
            y_matrix,
            E_matrix
        )

        best_llr = -np.inf

        for candidate_id in range(
            len(self.candidate_clusters)
        ):
            result = self._llr_from_aggregates(
                y_in=y_inside[candidate_id],
                E_in=E_inside[candidate_id],
                y_out=y_outside[candidate_id],
                E_out=E_outside[candidate_id],
                method=method
            )

            llr = result["LLR"]

            if (
                result["converged"]
                and np.isfinite(llr)
                and llr > best_llr
            ):
                best_llr = llr

        if not np.isfinite(best_llr):
            return np.nan

        return float(best_llr)

    def _find_ranked_llrs_from_arrays(
        self,
        y_matrix,
        E_matrix,
        method
    ):
        """
        Return the top nclust_max LLRs subject to spatial non-overlap.

        Rank 1 is the largest LLR. Rank 2 is the largest LLR among
        candidates not overlapping rank 1, and so on.
        """

        y_inside, E_inside, y_outside, E_outside = (
            self._prepare_candidate_aggregates(y_matrix, E_matrix)
        )

        candidate_values = []
        for candidate_id in range(len(self.candidate_clusters)):
            result = self._llr_from_aggregates(
                y_in=y_inside[candidate_id],
                E_in=E_inside[candidate_id],
                y_out=y_outside[candidate_id],
                E_out=E_outside[candidate_id],
                method=method,
            )
            llr = result["LLR"]
            if result["converged"] and np.isfinite(llr):
                candidate_values.append((float(llr), candidate_id))

        candidate_values.sort(key=lambda item: item[0], reverse=True)

        selected_counties = set()
        selected_llrs = []
        for llr, candidate_id in candidate_values:
            cluster_set = set(self.candidate_clusters[candidate_id])
            if cluster_set & selected_counties:
                continue
            selected_llrs.append(llr)
            selected_counties.update(cluster_set)
            if len(selected_llrs) >= self.nclust_max:
                break

        output = np.full(self.nclust_max, np.nan, dtype=float)
        output[:len(selected_llrs)] = selected_llrs
        return output

    def _find_top_llr_from_arrays(
        self,
        y_matrix,
        E_matrix,
        method
    ):
        """
        Find the best observed candidate cluster(s).

        When nclust_max=1, this avoids constructing and sorting a
        DataFrame containing every candidate.
        """

        (
            y_inside,
            E_inside,
            y_outside,
            E_outside
        ) = self._prepare_candidate_aggregates(
            y_matrix,
            E_matrix
        )

        # Fast single-cluster path
        if self.nclust_max == 1:

            best_candidate_id = None
            best_result = None
            best_llr = -np.inf

            for candidate_id in range(
                len(self.candidate_clusters)
            ):
                result = self._llr_from_aggregates(
                    y_in=y_inside[candidate_id],
                    E_in=E_inside[candidate_id],
                    y_out=y_outside[candidate_id],
                    E_out=E_outside[candidate_id],
                    method=method
                )

                llr = result["LLR"]

                if (
                    result["converged"]
                    and np.isfinite(llr)
                    and llr > best_llr
                ):
                    best_candidate_id = candidate_id
                    best_result = result
                    best_llr = llr

            if best_result is None:
                return pd.DataFrame([{
                    "rank": 1,
                    "candidate_id": None,
                    "method": method.name,
                    "LLR": 0.0,
                    "ll_alt": np.nan,
                    "ll_null": np.nan,
                    "converged": False,
                    "cluster": None,
                    "cluster_size": 0,
                    "population_at_risk": np.nan,
                    "population_percent": np.nan,
                }])

            best_cluster = list(
                self.candidate_clusters[
                    best_candidate_id
                ]
            )

            return pd.DataFrame([{
                "rank": 1,
                "candidate_id": best_candidate_id,
                "method": method.name,
                "LLR": float(best_llr),
                "ll_alt": best_result["ll_alt"],
                "ll_null": best_result["ll_null"],
                "converged": True,
                "cluster": best_cluster,
                "cluster_size": len(best_cluster),
                "population_at_risk": float(
                    self.candidate_population[best_candidate_id]
                ),
                "population_percent": float(
                    self.candidate_population_percent[best_candidate_id]
                ),
            }])

        # Multiple-cluster path
        candidate_results = []

        for candidate_id, cluster in enumerate(
            self.candidate_clusters
        ):
            result = self._llr_from_aggregates(
                y_in=y_inside[candidate_id],
                E_in=E_inside[candidate_id],
                y_out=y_outside[candidate_id],
                E_out=E_outside[candidate_id],
                method=method
            )

            if (
                result["converged"]
                and np.isfinite(result["LLR"])
            ):
                candidate_results.append({
                    "candidate_id": candidate_id,
                    "method": method.name,
                    "LLR": result["LLR"],
                    "ll_alt": result["ll_alt"],
                    "ll_null": result["ll_null"],
                    "converged": True,
                    "cluster": list(cluster),
                    "cluster_size": len(cluster),
                    "population_at_risk": float(
                        self.candidate_population[candidate_id]
                    ),
                    "population_percent": float(
                        self.candidate_population_percent[candidate_id]
                    ),
                })

        if not candidate_results:
            return pd.DataFrame([{
                "rank": 1,
                "candidate_id": None,
                "method": method.name,
                "LLR": 0.0,
                "ll_alt": np.nan,
                "ll_null": np.nan,
                "converged": False,
                "cluster": None,
                "cluster_size": 0,
                "population_at_risk": np.nan,
                "population_percent": np.nan,
            }])

        candidate_results.sort(
            key=lambda item: item["LLR"],
            reverse=True
        )

        selected = []
        selected_counties = set()

        for result in candidate_results:

            candidate_set = set(
                result["cluster"]
            )

            if candidate_set & selected_counties:
                continue

            selected.append(result)
            selected_counties.update(
                candidate_set
            )

            if len(selected) >= self.nclust_max:
                break

        result_df = pd.DataFrame(selected)
        result_df.insert(
            0,
            "rank",
            np.arange(1, len(result_df) + 1)
        )

        return result_df

    def _find_top_llr(
        self,
        df_sim,
        method
    ):
        """DataFrame-compatible wrapper for observed scans."""

        if len(self.candidate_clusters) == 0:
            self.create_candidate_clusters(population_data=df_sim)

        y_matrix, E_matrix = (
            self._prepare_scan_arrays(df_sim)
        )

        return self._find_top_llr_from_arrays(
            y_matrix=y_matrix,
            E_matrix=E_matrix,
            method=method
        )

    def _fit_global_null_probabilities(
        self,
        y_matrix,
        E_matrix,
        max_iter=10000,
        tol=1e-10,
    ):
        """
        Fit the conditional global null with county-specific intercepts.

        The global no-interaction model is

            log(mu_it) = log(E_it) + gamma_i + f_0(t).

        Conditional on the observed total N_t at time t, f_0(t) cancels and

            p_it = E_it * exp(gamma_i)
                   / sum_j E_jt * exp(gamma_j).

        The county multipliers exp(gamma_i) are estimated by iterative
        proportional fitting.  This is the conditional maximum-likelihood fit
        of the county main effects with the time margins held fixed.  The
        resulting fitted county totals match the observed county totals, up to
        the numerical tolerance.
        """

        y_matrix = np.asarray(y_matrix, dtype=float)
        E_matrix = np.asarray(E_matrix, dtype=float)

        expected_shape = (len(self.counties), len(self.years))
        if y_matrix.shape != expected_shape:
            raise ValueError(
                f"y_matrix has shape {y_matrix.shape}; "
                f"expected {expected_shape}."
            )
        if E_matrix.shape != expected_shape:
            raise ValueError(
                f"E_matrix has shape {E_matrix.shape}; "
                f"expected {expected_shape}."
            )
        if np.any(~np.isfinite(y_matrix)) or np.any(y_matrix < 0):
            raise ValueError("Observed counts must be finite and non-negative.")
        if np.any(~np.isfinite(E_matrix)) or np.any(E_matrix <= 0):
            raise ValueError("Expected counts must be finite and positive.")
        if not np.allclose(
            y_matrix,
            np.rint(y_matrix),
            rtol=0.0,
            atol=1e-8,
        ):
            raise ValueError("Observed Poisson counts must be integer-valued.")

        total_cases_by_time = y_matrix.sum(axis=0)
        observed_county_totals = y_matrix.sum(axis=1)

        if total_cases_by_time.sum() <= 0:
            raise ValueError("At least one case is required for the global null.")

        # Counties with a zero total have their conditional MLE on the
        # boundary, exp(gamma_i)=0.  Positive-total counties are fitted by
        # multiplicative iterative proportional fitting.
        active = observed_county_totals > 0
        if not np.any(active):
            raise ValueError("At least one county must contain observed cases.")

        risk_multipliers = np.ones(len(self.counties), dtype=float)
        risk_multipliers[~active] = 0.0
        converged = False

        for iteration in range(1, int(max_iter) + 1):
            weighted_expected = (
                E_matrix * risk_multipliers[:, None]
            )
            denominators = weighted_expected.sum(axis=0)

            if (
                np.any(~np.isfinite(denominators))
                or np.any(denominators <= 0)
            ):
                raise RuntimeError(
                    "The fitted global-null denominators are invalid."
                )

            probabilities = weighted_expected / denominators[None, :]
            fitted_mean = probabilities * total_cases_by_time[None, :]
            fitted_county_totals = fitted_mean.sum(axis=1)

            relative_error = np.max(
                np.abs(
                    fitted_county_totals[active]
                    - observed_county_totals[active]
                )
                / np.maximum(1.0, observed_county_totals[active])
            )

            if relative_error <= tol:
                converged = True
                break

            if np.any(fitted_county_totals[active] <= 0):
                raise RuntimeError(
                    "A positive-total county received a zero fitted total "
                    "under the global null."
                )

            risk_multipliers[active] *= (
                observed_county_totals[active]
                / fitted_county_totals[active]
            )

            # The intercepts are identifiable only up to a common additive
            # constant.  Centering their logarithms prevents numerical drift
            # without changing any conditional probability.
            log_active = np.log(risk_multipliers[active])
            risk_multipliers[active] = np.exp(
                log_active - log_active.mean()
            )

        if not converged:
            raise RuntimeError(
                "The fitted global-null county intercepts did not converge "
                f"within {max_iter} iterations."
            )

        # Recalculate once from the final multipliers retained at convergence.
        weighted_expected = E_matrix * risk_multipliers[:, None]
        probabilities = (
            weighted_expected
            / weighted_expected.sum(axis=0, keepdims=True)
        )
        fitted_mean = probabilities * total_cases_by_time[None, :]
        fitted_county_totals = fitted_mean.sum(axis=1)

        if not np.allclose(
            probabilities.sum(axis=0),
            1.0,
            rtol=1e-10,
            atol=1e-12,
        ):
            raise RuntimeError(
                "Fitted global-null probabilities do not sum to one."
            )

        max_margin_error = float(
            np.max(
                np.abs(
                    fitted_county_totals
                    - observed_county_totals
                )
            )
        )

        gamma = np.full(len(self.counties), -np.inf, dtype=float)
        gamma[active] = np.log(risk_multipliers[active])

        probabilities.setflags(write=False)
        fitted_mean.setflags(write=False)
        gamma.setflags(write=False)
        risk_multipliers.setflags(write=False)

        return {
            "probabilities": probabilities,
            "fitted_mean": fitted_mean,
            "gamma": gamma,
            "risk_multipliers": risk_multipliers,
            "observed_county_totals": observed_county_totals,
            "fitted_county_totals": fitted_county_totals,
            "total_cases_by_time": total_cases_by_time,
            "iterations": int(iteration),
            "max_county_margin_error": max_margin_error,
            "converged": True,
        }

    def _one_null_stat(
        self,
        y_matrix,
        E_matrix,
        null_probabilities,
        method,
        seed_b
    ):
        """
        Generate one conditional global-null dataset and return ranked LLRs.

        For every time point, the observed total number of cases is fixed and
        redistributed using

            p_it proportional to E_it * exp(gamma_i),

        where gamma_i is the fitted county-specific baseline intercept.  The
        common temporal component cancels after conditioning on the time total.
        """

        y_matrix = np.asarray(y_matrix, dtype=float)
        E_matrix = np.asarray(E_matrix, dtype=float)
        null_probabilities = np.asarray(
            null_probabilities,
            dtype=float,
        )

        expected_shape = (len(self.counties), len(self.years))

        if y_matrix.shape != expected_shape:
            raise ValueError(
                f"y_matrix has shape {y_matrix.shape}; "
                f"expected {expected_shape}."
            )
        if E_matrix.shape != expected_shape:
            raise ValueError(
                f"E_matrix has shape {E_matrix.shape}; "
                f"expected {expected_shape}."
            )
        if null_probabilities.shape != expected_shape:
            raise ValueError(
                "null_probabilities has shape "
                f"{null_probabilities.shape}; expected {expected_shape}."
            )
        if np.any(~np.isfinite(y_matrix)) or np.any(y_matrix < 0):
            raise ValueError("Observed counts must be finite and non-negative.")
        if np.any(~np.isfinite(E_matrix)) or np.any(E_matrix <= 0):
            raise ValueError("Expected counts must be finite and positive.")
        if (
            np.any(~np.isfinite(null_probabilities))
            or np.any(null_probabilities < 0)
        ):
            raise ValueError(
                "Global-null probabilities must be finite and non-negative."
            )
        if not np.allclose(
            null_probabilities.sum(axis=0),
            1.0,
            rtol=1e-10,
            atol=1e-12,
        ):
            raise ValueError(
                "Global-null probabilities must sum to one at every time."
            )

        total_cases_by_time = y_matrix.sum(axis=0)
        rounded_totals = np.rint(total_cases_by_time)

        if not np.allclose(
            total_cases_by_time,
            rounded_totals,
            rtol=0.0,
            atol=1e-8,
        ):
            raise ValueError(
                "Observed case totals must be integer-valued for multinomial "
                "null generation."
            )

        rng = np.random.default_rng(seed_b)
        n_counties, n_times = E_matrix.shape
        y_null_matrix = np.zeros((n_counties, n_times), dtype=float)

        for time_index in range(n_times):
            total_cases_t = int(rounded_totals[time_index])
            probabilities_t = null_probabilities[:, time_index]
            y_null_matrix[:, time_index] = rng.multinomial(
                n=total_cases_t,
                pvals=probabilities_t,
            )

        return self._find_ranked_llrs_from_arrays(
            y_matrix=y_null_matrix,
            E_matrix=E_matrix,
            method=method,
        )

    def _get_null_stats(
        self,
        y_matrix,
        E_matrix,
        null_probabilities,
        method,
        seed,
        n_jobs=1
    ):
        """
        Return a Bmc-by-nclust_max matrix of rank-specific null LLRs.

        n_jobs=1 is the safest option in Jupyter. Positive values greater
        than one use a thread backend rather than Windows process spawning.
        """

        y_matrix = np.asarray(y_matrix, dtype=float)
        E_matrix = np.asarray(E_matrix, dtype=float)
        null_probabilities = np.asarray(
            null_probabilities,
            dtype=float,
        )
        method_key = self._method_cache_key(method)

        cache_key = (
            method_key,
            int(seed),
            int(self.Bmc),
            int(self.nclust_max),
            y_matrix.shape,
            y_matrix.sum(axis=0).tobytes(),
            E_matrix.shape,
            E_matrix.tobytes(),
            null_probabilities.tobytes(),
            tuple(tuple(cluster) for cluster in self.candidate_clusters),
        )

        cached = self._null_stats_cache.get(cache_key)
        if cached is not None:
            return cached.copy()

        if n_jobs in (None, 0, -1):
            safe_jobs = 1
        else:
            safe_jobs = max(1, int(n_jobs))

        seeds = [int(seed) + replication + 1 for replication in range(self.Bmc)]

        if safe_jobs == 1:
            null_stats = [
                self._one_null_stat(
                    y_matrix=y_matrix,
                    E_matrix=E_matrix,
                    null_probabilities=null_probabilities,
                    method=method,
                    seed_b=seed_b,
                )
                for seed_b in seeds
            ]
        else:
            null_stats = Parallel(
                n_jobs=safe_jobs,
                prefer="threads",
                verbose=0,
            )(
                delayed(self._one_null_stat)(
                    y_matrix=y_matrix,
                    E_matrix=E_matrix,
                    null_probabilities=null_probabilities,
                    method=method,
                    seed_b=seed_b,
                )
                for seed_b in seeds
            )

        null_stats = np.asarray(null_stats, dtype=float)
        expected_shape = (self.Bmc, self.nclust_max)
        if null_stats.shape != expected_shape:
            raise RuntimeError(
                f"Null-statistic matrix has shape {null_stats.shape}; "
                f"expected {expected_shape}."
            )

        self._null_stats_cache[cache_key] = null_stats.copy()
        return null_stats

    def _calibrate_mc_cutoff(
        self,
        y_matrix,
        E_matrix,
        null_probabilities,
        method: MethodSpec,
        seed,
        n_jobs=1
    ):
        """Calibrate a separate Monte Carlo cutoff for every selected rank."""

        null_stats = self._get_null_stats(
            y_matrix=y_matrix,
            E_matrix=E_matrix,
            null_probabilities=null_probabilities,
            method=method,
            seed=int(seed),
            n_jobs=n_jobs,
        )

        finite_mask = np.isfinite(null_stats)
        n_mc_valid = finite_mask.sum(axis=0).astype(int)
        n_mc_failed = (~finite_mask).sum(axis=0).astype(int)

        if np.any(n_mc_failed > 0):
            failed = {
                rank + 1: int(value)
                for rank, value in enumerate(n_mc_failed)
                if value > 0
            }
            raise RuntimeError(
                "Some null replications could not produce the requested "
                f"number of non-overlapping clusters: {failed}. "
                "Reduce n_max/nclust_max or candidate min_size."
            )

        cutoffs = np.full(self.nclust_max, np.inf, dtype=float)

        for rank_index in range(self.nclust_max):
            rank_null = null_stats[:, rank_index]
            n_valid = int(n_mc_valid[rank_index])

            max_exceedances = int(
                np.floor(self.alpha * (n_valid + 1) - 1.0 + 1e-12)
            )
            if max_exceedances >= 0:
                ordered = np.sort(rank_null)
                cutoffs[rank_index] = float(
                    ordered[-(max_exceedances + 1)]
                )

        return {
            "null_stats": null_stats,
            "cutoffs": cutoffs,
            # Backward-compatible first-rank entries.
            "cutoff": float(cutoffs[0]),
            "n_mc_valid_by_rank": n_mc_valid,
            "n_mc_failed_by_rank": n_mc_failed,
            "n_mc_valid": int(n_mc_valid[0]),
            "n_mc_failed": int(n_mc_failed[0]),
        }

    def monte_carlo_pvalue_one_sim(
        self,
        df_sim,
        method: MethodSpec,
        calibration
    ):
        """Scan one dataset and calculate rank-specific Monte Carlo p-values."""

        y_matrix, E_matrix = self._prepare_scan_arrays(df_sim)
        observed_top = self._find_top_llr_from_arrays(
            y_matrix=y_matrix,
            E_matrix=E_matrix,
            method=method,
        )

        if observed_top.empty or not observed_top["converged"].any():
            return pd.DataFrame([{
                "method": method.name,
                "rank": 1,
                "T_obs": 0.0,
                "ll_alt": np.nan,
                "ll_null": np.nan,
                "converged": False,
                "p_value": 1.0,
                "mc_cutoff": float(calibration["cutoffs"][0]),
                "significant": False,
                "best_cluster": None,
                "cluster_size": 0,
                "population_at_risk": np.nan,
                "population_percent": np.nan,
                "n_mc_valid": int(calibration["n_mc_valid_by_rank"][0]),
                "n_mc_failed": int(calibration["n_mc_failed_by_rank"][0]),
            }])

        null_stats = np.asarray(calibration["null_stats"], dtype=float)
        cutoffs = np.asarray(calibration["cutoffs"], dtype=float)
        result_rows = []

        for observation in observed_top.itertuples(index=False):
            rank = int(observation.rank)
            rank_index = rank - 1
            if rank_index >= self.nclust_max:
                raise RuntimeError(
                    f"Observed rank {rank} exceeds nclust_max={self.nclust_max}."
                )

            rank_null = null_stats[:, rank_index]
            T_obs = float(observation.LLR)
            exceedances = int(np.count_nonzero(rank_null >= T_obs))
            p_value = (1 + exceedances) / (len(rank_null) + 1)
            cutoff = float(cutoffs[rank_index])
            significant = bool(T_obs > cutoff)

            if significant != bool(p_value <= self.alpha):
                raise RuntimeError(
                    "The fixed-cutoff and Monte Carlo p-value rules "
                    f"disagree for {method.name}, rank {rank}."
                )

            result_rows.append({
                "method": method.name,
                "rank": rank,
                "T_obs": T_obs,
                "ll_alt": observation.ll_alt,
                "ll_null": observation.ll_null,
                "converged": True,
                "p_value": float(p_value),
                "mc_cutoff": cutoff,
                "significant": significant,
                "best_cluster": observation.cluster,
                "cluster_size": int(observation.cluster_size),
                "population_at_risk": float(observation.population_at_risk),
                "population_percent": float(observation.population_percent),
                "n_mc_valid": int(
                    calibration["n_mc_valid_by_rank"][rank_index]
                ),
                "n_mc_failed": int(
                    calibration["n_mc_failed_by_rank"][rank_index]
                ),
            })

        return pd.DataFrame(result_rows)

    def monte_carlo_all_simulations(
        self,
        generated_data,
        methods=DEFAULT_METHODS,
        seed=None,
        n_jobs=1
    ):
        """Run every simulation using rank-specific multiple-cluster nulls."""

        methods = select_methods(methods)

        if seed is None:
            seed = self.seed

        grouped_simulations = list(generated_data.groupby("sim", sort=True))
        if not grouped_simulations:
            return pd.DataFrame()

        if len(self.candidate_clusters) == 0:
            self.create_candidate_clusters(
                population_data=grouped_simulations[0][1]
            )
        reference_candidate_population = (
            self.candidate_population_reference.copy()
        )

        result_frames = []
        for sim_id, df_sim in grouped_simulations:
            current_candidate_population = self._resolve_candidate_population(
                df_sim
            )
            if not np.allclose(
                current_candidate_population,
                reference_candidate_population,
                rtol=1e-10,
                atol=1e-8,
            ):
                raise ValueError(
                    "Population-at-risk values vary between simulations. "
                    "Run different population settings separately so every "
                    "simulation uses the intended population-limited windows."
                )

            y_matrix, E_matrix = self._prepare_scan_arrays(df_sim)
            global_null_fit = self._fit_global_null_probabilities(
                y_matrix=y_matrix,
                E_matrix=E_matrix,
            )
            null_probabilities = global_null_fit["probabilities"]

            for method_index, method in enumerate(methods):
                method_seed = (
                    int(seed)
                    + int(sim_id) * 1_000_000
                    + method_index * 10_000
                )
                calibration = self._calibrate_mc_cutoff(
                    y_matrix=y_matrix,
                    E_matrix=E_matrix,
                    null_probabilities=null_probabilities,
                    method=method,
                    seed=method_seed,
                    n_jobs=n_jobs,
                )

                result_df = self.monte_carlo_pvalue_one_sim(
                    df_sim=df_sim,
                    method=method,
                    calibration=calibration,
                )
                result_df.insert(0, "sim", int(sim_id))
                result_frames.append(result_df)

                # Do not retain a large calibration for all methods.
                del calibration

        if not result_frames:
            return pd.DataFrame()

        columns = [
            "sim", "method", "rank", "T_obs", "ll_alt", "ll_null",
            "converged", "p_value", "mc_cutoff", "significant",
            "best_cluster", "cluster_size", "population_at_risk",
            "population_percent", "n_mc_valid", "n_mc_failed",
        ]
        return pd.concat(result_frames, ignore_index=True)[columns]

    def compute_cluster_metrics(
        self,
        detected_cluster,
        true_cluster=None,
        all_counties=None
    ):
        """Calculate binary recovery metrics for one or many clusters."""

        if true_cluster is None:
            true_cluster = self.true_clusters
        if all_counties is None:
            all_counties = self.counties

        all_counties = np.asarray(all_counties, dtype=int)
        true_set = self._flatten_clusters(true_cluster)
        detected_set = self._flatten_clusters(detected_cluster)

        y_true = np.isin(all_counties, list(true_set))
        y_pred = np.isin(all_counties, list(detected_set))

        TP = int(np.count_nonzero(y_true & y_pred))
        TN = int(np.count_nonzero(~y_true & ~y_pred))
        FP = int(np.count_nonzero(~y_true & y_pred))
        FN = int(np.count_nonzero(y_true & ~y_pred))

        sensitivity = TP / (TP + FN) if TP + FN > 0 else 0.0
        specificity = TN / (TN + FP) if TN + FP > 0 else 0.0
        precision = TP / (TP + FP) if TP + FP > 0 else 0.0
        jaccard = TP / (TP + FP + FN) if TP + FP + FN > 0 else 0.0
        accuracy = (TP + TN) / len(all_counties)

        return {
            "TP": TP,
            "TN": TN,
            "FP": FP,
            "FN": FN,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "precision": precision,
            "jaccard": jaccard,
            "rand_index": float(rand_score(y_true, y_pred)),
            "adjusted_rand_index": float(
                adjusted_rand_score(y_true, y_pred)
            ),
            "purity": accuracy,
            "accuracy": accuracy,
            "exact_recovery": bool(true_set == detected_set),
        }

    def summarize_power_and_metrics(
        self,
        mc_df
    ):
        """
        Summarize one row per simulation-method, combining all significant
        non-overlapping clusters selected up to nclust_max.
        """

        if mc_df.empty:
            return pd.DataFrame()

        metric_rows = []
        for (sim_id, method_name), group in mc_df.groupby(
            ["sim", "method"], sort=False
        ):
            group = group.sort_values("rank").copy()
            valid = group["converged"].astype(bool)
            significant = valid & group["significant"].astype(bool)
            detected_clusters = group.loc[
                significant, "best_cluster"
            ].tolist()

            metrics = self.compute_cluster_metrics(
                detected_cluster=detected_clusters
            )
            rejected = bool(significant.any())

            metric_rows.append({
                "sim": int(sim_id),
                "method": method_name,
                "significant": rejected,
                "min_p_value": float(group["p_value"].min()),
                "T_obs_max": float(group["T_obs"].max()),
                "n_selected": int(valid.sum()),
                "n_significant": int(significant.sum()),
                "detected_clusters": detected_clusters,
                "jaccard_rej": metrics["jaccard"] if rejected else np.nan,
                **metrics,
            })

        metrics_df = pd.DataFrame(metric_rows)
        return (
            metrics_df
            .groupby("method", as_index=False)
            .agg(
                n_sim=("sim", "nunique"),
                n_rejected=("significant", "sum"),
                power=("significant", "mean"),
                mean_min_p_value=("min_p_value", "mean"),
                mean_T_obs_max=("T_obs_max", "mean"),
                mean_n_selected=("n_selected", "mean"),
                mean_n_significant=("n_significant", "mean"),
                sensitivity=("sensitivity", "mean"),
                specificity=("specificity", "mean"),
                precision=("precision", "mean"),
                jaccard=("jaccard", "mean"),
                jaccard_rej=("jaccard_rej", "mean"),
                rand_index=("rand_index", "mean"),
                adjusted_rand_index=("adjusted_rand_index", "mean"),
                purity=("purity", "mean"),
                accuracy=("accuracy", "mean"),
                exact_recovery=("exact_recovery", "mean"),
            )
        )

    def summarize_type1_error(
    self,
    mc_null_df
    ):
        
        if mc_null_df.empty:
            return pd.DataFrame()

        data = mc_null_df[
            [
                "sim",
                "method",
                "p_value",
                "significant",
                "converged"
            ]
        ].copy()

        # Use only valid converged simulations
        data = data[
            data["converged"]
        ].copy()

        data["false_rejection"] = (
            data["significant"].astype(bool)
        )

        summary = (
            data
            .groupby(
                "method",
                as_index=False
            )
            .agg(
                n_null_sim=("sim", "nunique"),

                n_false_rejections=(
                    "false_rejection",
                    "sum"
                ),

                type1_error=(
                    "false_rejection",
                    "mean"
                ),

                null_mean_p_value=(
                    "p_value",
                    "mean"
                )
            )
        )

        summary["type1_se"] = np.sqrt(
            summary["type1_error"]
            * (1.0 - summary["type1_error"])
            / summary["n_null_sim"]
        )

        return summary




# ======================================================================
# SAVED-DATA BATCH UTILITIES
# ======================================================================


def _natural_path_key(path: Path):
    """Sort file names naturally: sim_2.csv before sim_10.csv."""

    return [
        int(piece) if piece.isdigit() else piece.lower()
        for piece in re.split(r"(\d+)", path.name)
    ]


def _normalise_saved_dataset_columns(
    data: pd.DataFrame,
    source_path: Path | None = None,
) -> pd.DataFrame:
    """
    Normalize one saved simulation file.

    Required analysis columns after normalization:
        county, year, population, observed_cases

    Population is read directly from every county-year row. It is never
    replaced by a fixed value such as 10,000.
    """

    data = data.copy()
    data.columns = [str(column).strip() for column in data.columns]

    aliases = {
        "cases": "observed_cases",
        "case": "observed_cases",
        "observed": "observed_cases",
        "observed_case": "observed_cases",
        "pop": "population",
    }

    lower_to_actual = {
        str(column).strip().lower(): column
        for column in data.columns
    }

    for alias, target in aliases.items():
        actual_alias = lower_to_actual.get(alias)
        actual_target = lower_to_actual.get(target)
        if actual_alias is not None and actual_target is None:
            data = data.rename(columns={actual_alias: target})
            lower_to_actual[target] = target

    required = {
        "county",
        "year",
        "population",
        "observed_cases",
    }
    missing = required - set(data.columns)
    if missing:
        location = f" in {source_path}" if source_path is not None else ""
        raise ValueError(
            f"Missing required columns{location}: {sorted(missing)}"
        )

    for column in ("county", "year", "population", "observed_cases"):
        data[column] = pd.to_numeric(data[column], errors="raise")

    data["county"] = data["county"].astype(int)
    data["year"] = data["year"].astype(int)
    data["population"] = data["population"].astype(float)
    data["observed_cases"] = data["observed_cases"].astype(float)

    if not np.all(np.isfinite(data["population"])):
        raise ValueError(f"Population contains non-finite values in {source_path}.")
    if np.any(data["population"] <= 0):
        raise ValueError(f"Population must be positive in {source_path}.")
    if not np.all(np.isfinite(data["observed_cases"])):
        raise ValueError(
            f"Observed cases contain non-finite values in {source_path}."
        )
    if np.any(data["observed_cases"] < 0):
        raise ValueError(f"Observed cases cannot be negative in {source_path}.")

    duplicate_mask = data.duplicated(["county", "year"], keep=False)
    if duplicate_mask.any():
        duplicate_rows = (
            data.loc[duplicate_mask, ["county", "year"]]
            .drop_duplicates()
            .head(10)
            .to_dict("records")
        )
        raise ValueError(
            f"Duplicate county-year rows in {source_path}: {duplicate_rows}"
        )

    return (
        data
        .sort_values(["county", "year"])
        .reset_index(drop=True)
    )


def _as_cluster_mask(values: pd.Series) -> np.ndarray:
    """Convert common TRUE/FALSE, 1/0, and string encodings to a mask."""

    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).to_numpy(dtype=bool)

    numeric = pd.to_numeric(values, errors="coerce")
    numeric_known = numeric.notna()

    result = np.zeros(len(values), dtype=bool)
    if numeric_known.any():
        result[numeric_known.to_numpy()] = (
            numeric.loc[numeric_known].to_numpy(dtype=float) > 0
        )

    unknown = ~numeric_known
    if unknown.any():
        normalized = (
            values.loc[unknown]
            .astype(str)
            .str.strip()
            .str.lower()
        )
        true_tokens = {
            "true", "t", "yes", "y", "inside", "cluster", "1"
        }
        result[unknown.to_numpy()] = normalized.isin(true_tokens).to_numpy()

    return result


def infer_true_clusters_from_saved_data(
    data: pd.DataFrame,
) -> list[list[int]] | None:
    """
    Infer the true cluster definition when the saved file includes truth.

    Preferred format:
        cluster_id = 0 outside, 1..K for true clusters.

    Fallback format:
        cluster = TRUE/FALSE or 1/0 for one true cluster.
    """

    if "cluster_id" in data.columns:
        cluster_ids = pd.to_numeric(data["cluster_id"], errors="coerce")
        positive_ids = sorted(
            int(value)
            for value in cluster_ids.dropna().unique()
            if float(value) > 0
        )

        clusters = []
        for cluster_id in positive_ids:
            counties = sorted(
                data.loc[
                    cluster_ids == cluster_id,
                    "county",
                ]
                .astype(int)
                .unique()
                .tolist()
            )
            if counties:
                clusters.append(counties)

        if clusters:
            return clusters

    if "cluster" in data.columns:
        mask = _as_cluster_mask(data["cluster"])
        counties = sorted(
            data.loc[mask, "county"]
            .astype(int)
            .unique()
            .tolist()
        )
        if counties:
            return [counties]

    return None


def _canonical_clusters(
    clusters: Sequence[Sequence[int]] | Sequence[int] | None,
) -> tuple[tuple[int, ...], ...]:
    """Canonical representation used to compare truth across files."""

    if clusters is None:
        return tuple()

    values = list(clusters)
    if not values:
        return tuple()

    if not isinstance(
        values[0],
        (list, tuple, set, np.ndarray, pd.Series),
    ):
        values = [values]

    normalized = [
        tuple(sorted(int(county) for county in cluster))
        for cluster in values
    ]
    return tuple(sorted(normalized))


def _parse_cluster_cell(value):
    """Restore list-valued best_cluster cells after reading a checkpoint CSV."""

    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    if isinstance(value, (list, tuple, set, np.ndarray)):
        return [int(item) for item in value]

    text = str(value).strip()
    if not text or text.lower() in {"none", "nan"}:
        return None

    try:
        parsed = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return None

    if parsed is None:
        return None
    return [int(item) for item in np.atleast_1d(parsed).tolist()]


def discover_saved_simulation_files(
    data_dir,
    n_files=100,
    pattern="*.csv",
    require_exact_count=True,
) -> list[Path]:
    """Return the first n_files in natural filename order."""

    data_dir = Path(data_dir)
    if not data_dir.exists():
        raise FileNotFoundError(f"Data directory does not exist: {data_dir}")
    if not data_dir.is_dir():
        raise NotADirectoryError(f"Not a directory: {data_dir}")

    files = sorted(data_dir.glob(pattern), key=_natural_path_key)

    if require_exact_count and len(files) < n_files:
        raise FileNotFoundError(
            f"Requested {n_files} files, but only {len(files)} matching "
            f"{pattern!r} were found in {data_dir}."
        )

    selected = files[:n_files]
    if not selected:
        raise FileNotFoundError(
            f"No files matching {pattern!r} were found in {data_dir}."
        )

    return selected



def _atomic_write_csv(dataframe, path):
    """Write a CSV checkpoint atomically so interruption cannot corrupt it."""

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + ".tmp")
    dataframe.to_csv(temporary_path, index=False)
    os.replace(temporary_path, path)


def _read_result_checkpoint(path):
    """Read one result checkpoint and restore list-valued cluster columns."""

    dataframe = pd.read_csv(path)
    if "best_cluster" in dataframe.columns:
        dataframe["best_cluster"] = dataframe["best_cluster"].apply(
            _parse_cluster_cell
        )
    return dataframe


def _result_has_all_methods(result_df, methods):
    """Return True for a complete checkpoint from this implementation."""

    if result_df is None or result_df.empty or "method" not in result_df.columns:
        return False

    # Earlier versions use county-count limits or time-specific rates.
    # Their likelihoods/checkpoints are incompatible with this pooled offset.
    if "implementation_version" not in result_df.columns:
        return False
    versions = set(result_df["implementation_version"].astype(str).unique())
    if versions != {IMPLEMENTATION_VERSION}:
        return False

    if "expected_count_definition" not in result_df.columns:
        return False
    definitions = set(result_df["expected_count_definition"].astype(str).unique())
    if definitions != {EXPECTED_COUNT_DEFINITION}:
        return False

    expected_methods = {method.name for method in methods}
    completed_methods = set(result_df["method"].astype(str).unique())
    return expected_methods.issubset(completed_methods)


def _build_rank_summary(all_results):
    """Create the rank-specific summary used by the batch runner."""

    if all_results.empty:
        return pd.DataFrame()

    return (
        all_results
        .groupby(["method", "rank"], as_index=False)
        .agg(
            n_files=("sim", "nunique"),
            n_converged=("converged", "sum"),
            n_significant=("significant", "sum"),
            rejection_rate=("significant", "mean"),
            mean_p_value=("p_value", "mean"),
            mean_T_obs=("T_obs", "mean"),
            mean_cluster_size=("cluster_size", "mean"),
            mean_population_percent=("population_percent", "mean"),
            max_population_percent=("population_percent", "max"),
        )
    )


def _load_completed_file_results(per_file_dir, methods):
    """Load only file-level checkpoints containing every requested method."""

    completed_frames = []

    for result_path in sorted(
        Path(per_file_dir).glob("*_svtt_results.csv"),
        key=_natural_path_key,
    ):
        try:
            result_df = _read_result_checkpoint(result_path)
        except Exception:
            # A malformed checkpoint is ignored and will be regenerated when
            # its original input file is encountered again.
            continue

        if _result_has_all_methods(result_df, methods):
            completed_frames.append(result_df)

    if not completed_frames:
        return pd.DataFrame()

    return pd.concat(completed_frames, ignore_index=True)


def _save_live_batch_outputs(
    scanner,
    per_file_dir,
    output_dir,
    methods,
):
    """
    Rebuild and save cumulative batch outputs from completed file checkpoints.

    This is called after every completed input file. Therefore a kernel stop
    loses at most the currently running method, not all previous files.
    """

    all_results = _load_completed_file_results(
        per_file_dir=per_file_dir,
        methods=methods,
    )

    if all_results.empty:
        return {
            "results": pd.DataFrame(),
            "summary": pd.DataFrame(),
            "rank_summary": pd.DataFrame(),
        }

    output_dir = Path(output_dir)

    # Save a continuously updated result file while the batch is running.
    _atomic_write_csv(
        all_results,
        output_dir / "all_completed_files_mc_results.csv",
    )

    summary_input = all_results.copy()
    if "best_cluster" in summary_input.columns:
        summary_input["best_cluster"] = summary_input["best_cluster"].apply(
            _parse_cluster_cell
        )

    summary = scanner.summarize_power_and_metrics(summary_input)
    _atomic_write_csv(
        summary,
        output_dir / "power_and_cluster_metrics_summary_live.csv",
    )

    rank_summary = _build_rank_summary(all_results)
    _atomic_write_csv(
        rank_summary,
        output_dir / "rank_specific_summary_live.csv",
    )

    return {
        "results": all_results,
        "summary": summary,
        "rank_summary": rank_summary,
    }


def run_100_saved_files(
    data_dir,
    coord_path,
    output_dir,
    true_cluster=None,
    n_files=100,
    n_max=1,
    methods=DEFAULT_METHODS,
    Bmc=199,
    alpha=0.05,
    min_size=1,
    max_size_percent=25.0,
    step=1,
    seed=218720,
    n_jobs=1,
    file_pattern="*.csv",
    resume=True,
    stop_on_error=False,
):
    """
    Run SVTT on saved simulation files with method-level checkpointing.

    Resume behavior
    ---------------
    1. Every method is saved separately for every input CSV.
    2. The complete result for an input CSV is rebuilt after each method.
    3. Cumulative batch results and summaries are updated after each file.
    4. On restart, completed methods, files, and summaries are reused.

    Population is always read from the population column of each county-year
    row. No fixed population value is imposed on saved data. Candidate zones
    are restricted by their share of mean study-period population (the same
    as person-time share), not by the percentage of counties.
    """

    methods = select_methods(methods)

    if int(n_jobs) != 1:
        print(
            "For kernel safety, n_jobs has been changed to 1. "
            "The batch will run sequentially."
        )
    n_jobs = 1

    files = discover_saved_simulation_files(
        data_dir=data_dir,
        n_files=int(n_files),
        pattern=file_pattern,
        require_exact_count=True,
    )

    output_dir = Path(output_dir)
    per_file_dir = output_dir / "per_file_results"
    output_dir.mkdir(parents=True, exist_ok=True)
    per_file_dir.mkdir(parents=True, exist_ok=True)

    first_data = _normalise_saved_dataset_columns(
        pd.read_csv(files[0]),
        source_path=files[0],
    )

    study_years = np.sort(first_data["year"].unique().astype(int))
    if len(study_years) < 2:
        raise ValueError("At least two study years are required.")
    if not np.array_equal(
        study_years,
        np.arange(study_years.min(), study_years.max() + 1),
    ):
        raise ValueError(
            "Study years must be consecutive because the current scanner "
            "uses T_start:T_end."
        )

    inferred_truth = infer_true_clusters_from_saved_data(first_data)
    if true_cluster is None:
        if inferred_truth is None:
            raise ValueError(
                "true_cluster was not supplied and could not be inferred. "
                "Add cluster/cluster_id to the CSV files or set TRUE_CLUSTER."
            )
        true_cluster = inferred_truth

    expected_truth = _canonical_clusters(true_cluster)
    expected_counties = np.sort(first_data["county"].unique().astype(int))

    scanner = SVTT_Scan(
        coord_path=str(coord_path),
        T_start=int(study_years.min()),
        T_end=int(study_years.max()),
        Bmc=int(Bmc),
        true_cluster=true_cluster,
        alpha=float(alpha),
        min_size=int(min_size),
        max_size_percent=float(max_size_percent),
        step=int(step),
        seed=int(seed),
        n_max=int(n_max),
    )

    if not np.array_equal(expected_counties, scanner.counties):
        missing_from_data = sorted(
            set(scanner.counties.tolist()) - set(expected_counties.tolist())
        )
        extra_in_data = sorted(
            set(expected_counties.tolist()) - set(scanner.counties.tolist())
        )
        raise ValueError(
            "The first dataset and coordinate file contain different counties. "
            f"Missing from data: {missing_from_data}; "
            f"extra in data: {extra_in_data}."
        )

    # This is the critical population-window step.  County populations are
    # averaged over time (equivalently, converted to person-time shares) and
    # every retained candidate must contain no more than q percent of the
    # total population at risk.
    scanner.create_candidate_clusters(population_data=first_data)
    reference_candidate_population = (
        scanner.candidate_population_reference.copy()
    )

    manifest_path = output_dir / "file_manifest.csv"
    failed_path = output_dir / "failed_files.csv"
    progress_path = output_dir / "method_progress.csv"

    if resume and manifest_path.exists():
        manifest_rows = pd.read_csv(manifest_path).to_dict("records")
    else:
        manifest_rows = []

    if resume and failed_path.exists():
        failed_rows = pd.read_csv(failed_path).to_dict("records")
    else:
        failed_rows = []

    print("=" * 90)
    print("SAVED-DATA SVTT BATCH WITH METHOD CHECKPOINTS")
    print("=" * 90)
    print(f"Input directory : {Path(data_dir)}")
    print(f"Files selected  : {len(files)}")
    print(f"Study years     : {study_years.min()}--{study_years.max()}")
    print(f"True cluster(s) : {true_cluster}")
    print(f"n_max           : {scanner.nclust_max}")
    print(f"Candidates      : {len(scanner.candidate_clusters)}")
    print(
        "Window limit    : population share <= "
        f"{scanner.max_size_percent:g}%"
    )
    print(
        "Largest window  : "
        f"{scanner.candidate_population_percent.max():.6f}%"
    )
    print("Expected counts : population_it * pooled R0 (all locations and times)")
    print("Monte Carlo     : multinomial by time; fitted county intercepts")
    print(f"Bmc             : {scanner.Bmc}")
    print(f"Methods         : {[method.name for method in methods]}")
    print("Population      : read from each CSV county-year row")
    print("Checkpoint      : after every method and every input file")
    print("=" * 90)

    for simulation_id, file_path in enumerate(files, start=1):
        file_checkpoint_dir = (
            per_file_dir / f"{simulation_id:03d}_{file_path.stem}"
        )
        method_checkpoint_dir = file_checkpoint_dir / "methods"
        file_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        method_checkpoint_dir.mkdir(parents=True, exist_ok=True)

        result_path = (
            per_file_dir
            / f"{simulation_id:03d}_{file_path.stem}_svtt_results.csv"
        )
        status_path = file_checkpoint_dir / "status.csv"

        print("\n" + "-" * 90)
        print(
            f"FILE {simulation_id:03d}/{len(files):03d}: "
            f"{file_path.name}"
        )
        print("-" * 90)

        try:
            data = _normalise_saved_dataset_columns(
                pd.read_csv(file_path),
                source_path=file_path,
            )

            current_years = np.sort(data["year"].unique().astype(int))
            current_counties = np.sort(data["county"].unique().astype(int))

            if not np.array_equal(current_years, study_years):
                raise ValueError(
                    f"Years differ from the first file: {current_years.tolist()}"
                )
            if not np.array_equal(current_counties, expected_counties):
                raise ValueError("County set differs from the first file.")

            current_candidate_population = (
                scanner._resolve_candidate_population(data)
            )
            if not np.allclose(
                current_candidate_population,
                reference_candidate_population,
                rtol=1e-10,
                atol=1e-8,
            ):
                raise ValueError(
                    "County population-at-risk values differ from the first "
                    "simulation file. Candidate windows would therefore have "
                    "different population percentages. Run such population "
                    "settings in separate output folders."
                )

            current_truth = infer_true_clusters_from_saved_data(data)
            if (
                current_truth is not None
                and _canonical_clusters(current_truth) != expected_truth
            ):
                raise ValueError(
                    "The true-cluster definition differs from the first file. "
                    f"Found {current_truth}; expected {true_cluster}."
                )

            data["sim"] = int(simulation_id)

            standardized = scanner.standardize_saved_data(data)
            _atomic_write_csv(
                standardized,
                output_dir / "sir_diagnostics" / f"{file_path.stem}_fixed_R0_SIR.csv",
            )
            current_pooled_R0 = scanner.pooled_R0
            print(f"  Pooled R0   : {current_pooled_R0:.10g}")

            # Fast file-level resume: skip a file only when every requested
            # method is present in its aggregate checkpoint.
            if resume and result_path.exists():
                existing_result = _read_result_checkpoint(result_path)
            else:
                existing_result = pd.DataFrame()

            if _result_has_all_methods(existing_result, methods):
                print("Complete file checkpoint found; all methods skipped.")
                file_result_df = existing_result
            else:
                # Expected counts and the fitted global-null probabilities are
                # prepared once and reused for every temporal method.
                y_matrix, E_matrix = scanner._prepare_scan_arrays(data)
                global_null_fit = (
                    scanner._fit_global_null_probabilities(
                        y_matrix=y_matrix,
                        E_matrix=E_matrix,
                    )
                )
                null_probabilities = global_null_fit["probabilities"]

                print(
                    "  Global null : fitted county intercepts; "
                    f"iterations={global_null_fit['iterations']}, "
                    "max county-margin error="
                    f"{global_null_fit['max_county_margin_error']:.3e}"
                )

                method_frames = []
                progress_rows = []

                for method_index, method in enumerate(methods):
                    method_result_path = (
                        method_checkpoint_dir / f"{method.name}_results.csv"
                    )

                    checkpoint_is_current = False
                    if resume and method_result_path.exists():
                        checkpoint_candidate = _read_result_checkpoint(
                            method_result_path
                        )
                        checkpoint_is_current = _result_has_all_methods(
                            checkpoint_candidate,
                            [method],
                        )

                    if checkpoint_is_current:
                        method_result = checkpoint_candidate
                        print(
                            f"  {method.name:<10}: current checkpoint found; "
                            "skipped"
                        )
                    else:
                        print(f"  {method.name:<10}: running")

                        method_seed = (
                            int(seed)
                            + int(simulation_id) * 1_000_000
                            + method_index * 10_000
                        )

                        calibration = scanner._calibrate_mc_cutoff(
                            y_matrix=y_matrix,
                            E_matrix=E_matrix,
                            null_probabilities=null_probabilities,
                            method=method,
                            seed=method_seed,
                            n_jobs=n_jobs,
                        )

                        method_result = scanner.monte_carlo_pvalue_one_sim(
                            df_sim=data,
                            method=method,
                            calibration=calibration,
                        )

                        method_result.insert(0, "sim", int(simulation_id))
                        method_result.insert(1, "file_name", file_path.name)
                        method_result.insert(2, "file_path", str(file_path))
                        method_result.insert(
                            3,
                            "implementation_version",
                            IMPLEMENTATION_VERSION,
                        )
                        method_result.insert(
                            4,
                            "window_size_measure",
                            WINDOW_SIZE_MEASURE,
                        )
                        method_result.insert(
                            5,
                            "max_size_percent",
                            float(scanner.max_size_percent),
                        )

                        method_result["expected_count_definition"] = EXPECTED_COUNT_DEFINITION
                        method_result["null_calibration"] = NULL_CALIBRATION
                        method_result["pooled_R0"] = current_pooled_R0

                        _atomic_write_csv(
                            method_result,
                            method_result_path,
                        )
                        print(
                            f"  {method.name:<10}: checkpoint saved"
                        )

                        del calibration

                    # Normalize identifying columns for old/resumed checkpoints.
                    if "sim" not in method_result.columns:
                        method_result.insert(0, "sim", int(simulation_id))
                    else:
                        method_result["sim"] = int(simulation_id)
                    if "file_name" not in method_result.columns:
                        method_result.insert(1, "file_name", file_path.name)
                    if "file_path" not in method_result.columns:
                        method_result.insert(2, "file_path", str(file_path))

                    method_frames.append(method_result)
                    progress_rows.append({
                        "sim": int(simulation_id),
                        "file_name": file_path.name,
                        "method": method.name,
                        "status": "completed",
                        "checkpoint": str(method_result_path),
                    })

                    # Rebuild the file-level checkpoint after every method.
                    file_result_df = pd.concat(
                        method_frames,
                        ignore_index=True,
                    )
                    _atomic_write_csv(file_result_df, result_path)
                    _atomic_write_csv(pd.DataFrame(progress_rows), status_path)

                    # Global progress is also saved after every method.
                    global_progress = []
                    for status_file in sorted(
                        per_file_dir.glob("*/status.csv"),
                        key=_natural_path_key,
                    ):
                        try:
                            global_progress.append(pd.read_csv(status_file))
                        except Exception:
                            continue
                    if global_progress:
                        _atomic_write_csv(
                            pd.concat(global_progress, ignore_index=True),
                            progress_path,
                        )

                    scanner._null_stats_cache.clear()
                    gc.collect()

            if not _result_has_all_methods(file_result_df, methods):
                completed_names = sorted(
                    set(file_result_df.get("method", pd.Series(dtype=str)))
                )
                raise RuntimeError(
                    "The file checkpoint is incomplete after processing. "
                    f"Completed methods: {completed_names}."
                )

            # Replace any earlier manifest row for this simulation.
            manifest_rows = [
                row for row in manifest_rows
                if int(row.get("sim", -1)) != int(simulation_id)
            ]
            manifest_rows.append({
                "sim": int(simulation_id),
                "file_name": file_path.name,
                "file_path": str(file_path),
                "n_rows": int(len(data)),
                "n_counties": int(data["county"].nunique()),
                "n_years": int(data["year"].nunique()),
                "population_min": float(data["population"].min()),
                "population_max": float(data["population"].max()),
                "population_mean": float(data["population"].mean()),
                "population_varies": bool(data["population"].nunique() > 1),
                "implementation_version": IMPLEMENTATION_VERSION,
                "expected_count_definition": EXPECTED_COUNT_DEFINITION,
                "pooled_R0": current_pooled_R0,
                "null_calibration": NULL_CALIBRATION,
                "window_size_measure": WINDOW_SIZE_MEASURE,
                "max_size_percent": float(scanner.max_size_percent),
                "largest_candidate_population_percent": float(
                    scanner.candidate_population_percent.max()
                ),
                "methods_completed": len(methods),
                "status": "completed",
                "result_path": str(result_path),
            })
            _atomic_write_csv(pd.DataFrame(manifest_rows), manifest_path)

            # Remove an older failure entry when this file later succeeds.
            failed_rows = [
                row for row in failed_rows
                if int(row.get("sim", -1)) != int(simulation_id)
            ]
            if failed_rows:
                _atomic_write_csv(pd.DataFrame(failed_rows), failed_path)
            elif failed_path.exists():
                failed_path.unlink()

            live = _save_live_batch_outputs(
                scanner=scanner,
                per_file_dir=per_file_dir,
                output_dir=output_dir,
                methods=methods,
            )

            completed_file_count = (
                int(live["results"]["sim"].nunique())
                if not live["results"].empty
                else 0
            )
            print(
                "File complete. Cumulative completed files: "
                f"{completed_file_count}/{len(files)}"
            )
            print(
                "Population range: "
                f"{data['population'].min():,.2f}--"
                f"{data['population'].max():,.2f}"
            )

        except Exception as error:
            error_text = f"{type(error).__name__}: {error}"
            print(f"FAILED: {error_text}")

            failed_rows = [
                row for row in failed_rows
                if int(row.get("sim", -1)) != int(simulation_id)
            ]
            failed_rows.append({
                "sim": int(simulation_id),
                "file_name": file_path.name,
                "file_path": str(file_path),
                "error": error_text,
                "traceback": traceback.format_exc(),
            })
            _atomic_write_csv(pd.DataFrame(failed_rows), failed_path)

            manifest_rows = [
                row for row in manifest_rows
                if int(row.get("sim", -1)) != int(simulation_id)
            ]
            manifest_rows.append({
                "sim": int(simulation_id),
                "file_name": file_path.name,
                "file_path": str(file_path),
                "status": "failed",
                "error": error_text,
                "result_path": str(result_path),
            })
            _atomic_write_csv(pd.DataFrame(manifest_rows), manifest_path)

            if stop_on_error:
                raise

        finally:
            scanner._null_stats_cache.clear()
            gc.collect()

    final_outputs = _save_live_batch_outputs(
        scanner=scanner,
        per_file_dir=per_file_dir,
        output_dir=output_dir,
        methods=methods,
    )

    all_results = final_outputs["results"]
    summary = final_outputs["summary"]
    rank_summary = final_outputs["rank_summary"]

    if all_results.empty:
        raise RuntimeError(
            "No file completed successfully. See failed_files.csv."
        )

    _atomic_write_csv(
        all_results,
        output_dir / "all_100_files_mc_results.csv",
    )
    _atomic_write_csv(
        summary,
        output_dir / "power_and_cluster_metrics_summary.csv",
    )
    _atomic_write_csv(
        rank_summary,
        output_dir / "rank_specific_summary.csv",
    )

    batch_complete = all_results["sim"].nunique() == len(files)
    if batch_complete:
        marker_path = output_dir / "_BATCH_COMPLETE.txt"
        marker_path.write_text(
            "All requested files and methods completed successfully.\n",
            encoding="utf-8",
        )

    print("\n" + "=" * 90)
    print("BATCH FINISHED")
    print("=" * 90)
    print(f"Successful files: {all_results['sim'].nunique()}")
    print(f"Requested files : {len(files)}")
    print(f"Failed files    : {len(failed_rows)}")
    print(f"Output directory: {output_dir}")
    print("\nPower and cluster-recovery summary:")
    print(summary.to_string(index=False))

    return {
        "scanner": scanner,
        "files": files,
        "results": all_results,
        "summary": summary,
        "rank_summary": rank_summary,
        "manifest": pd.DataFrame(manifest_rows),
        "failed": pd.DataFrame(failed_rows),
        "output_dir": output_dir,
        "batch_complete": batch_complete,
    }


# ======================================================================


In [4]:
"""Run after svtt_fixed_r0.py in a Jupyter definition cell or with %run -i.

Processes the four two-cluster folders generated in the current workflow.
"""
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display

required_names = ("SVTT_Scan", "run_100_saved_files", "select_methods",
                  "_natural_path_key", "IMPLEMENTATION_VERSION",
                  "EXPECTED_COUNT_DEFINITION")
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError("Run svtt_fixed_r0.py first. Missing: " + ", ".join(missing))
if IMPLEMENTATION_VERSION != "2026-10-01-fixed-R0-population-v1":
    raise RuntimeError("The notebook still has the old scanner. Run svtt_fixed_r0.py first.")

SIM_ROOT = Path(r"C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data")
# Reuse a coordinate path already defined in the notebook.
if "COORD_PATH" not in globals():
    COORD_PATH = Path(
        r"C:\Users\ansarims\Desktop\RK_Summer1_2026"
        r"\Code\SVTT_24Aug2026\nm_county_coordinates.txt"
    )
COORD_PATH = Path(COORD_PATH)

N_FILES = 100
N_MAX = 2
BMC = 100
ALPHA = 0.05
MIN_SIZE = 1
MAX_SIZE_PERCENT = 25.0
STEP = 1
SEED = 218720
RESUME = True
ACTIVE_METHOD_NAMES = ("linear", "quadratic", "poly3", "poly4", "spline3", "spline4")
METHODS_TO_RUN = select_methods(ACTIVE_METHOD_NAMES)
TRUE_CLUSTERS = [[3, 15, 31], [12, 21, 32]]
SVTT_FILE_PATTERN = "sim_[0-9][0-9][0-9].csv"

configs = [
    (scenario, population)
    for scenario in ("Scenario_1", "Scenario_2")
    for population in ("Constant", "Original_Population")
]

# Settings-specific output folders prevent reusing a different analysis.
settings = dict(
    implementation_version=IMPLEMENTATION_VERSION,
    expected_count_definition=EXPECTED_COUNT_DEFINITION,
    Bmc=BMC, alpha=ALPHA, min_size=MIN_SIZE,
    max_size_percent=MAX_SIZE_PERCENT, step=STEP, seed=SEED,
    n_files=N_FILES, n_max=N_MAX, methods=ACTIVE_METHOD_NAMES,
    true_clusters=TRUE_CLUSTERS,
)
settings_hash = hashlib.sha256(
    json.dumps(settings, sort_keys=True).encode("utf-8")
).hexdigest()[:10]
RESULT_ROOT = SIM_ROOT / f"SVTT_fixed_R0_q{MAX_SIZE_PERCENT:g}_B{BMC}_{settings_hash}"

if not COORD_PATH.is_file():
    raise FileNotFoundError(f"Coordinate file not found: {COORD_PATH}")

# Check all four folders before starting the first batch.
for scenario, population in configs:
    input_dir = SIM_ROOT / "Two_Cluster" / scenario / population
    if not input_dir.is_dir():
        raise FileNotFoundError(f"Input folder not found: {input_dir}")
    matching = sorted(input_dir.glob(SVTT_FILE_PATTERN), key=_natural_path_key)
    if len(matching) < N_FILES:
        raise ValueError(f"{input_dir}: found {len(matching)} datasets; need {N_FILES}.")
    print(f"{scenario}, {population}: {len(matching)} datasets available.")

two_cluster_batch_results = {}
summaries = []
for scenario, population in configs:
    input_dir = SIM_ROOT / "Two_Cluster" / scenario / population
    output_dir = RESULT_ROOT / scenario / population
    print(f"\nRUNNING {scenario}: {population}")
    print("Output:", output_dir)

    batch = run_100_saved_files(
        data_dir=input_dir, coord_path=COORD_PATH, output_dir=output_dir,
        true_cluster=TRUE_CLUSTERS, n_files=N_FILES, n_max=N_MAX,
        methods=METHODS_TO_RUN, Bmc=BMC, alpha=ALPHA,
        min_size=MIN_SIZE, max_size_percent=MAX_SIZE_PERCENT,
        step=STEP, seed=SEED, n_jobs=1,
        file_pattern=SVTT_FILE_PATTERN, resume=RESUME, stop_on_error=False,
    )
    two_cluster_batch_results[(scenario, population)] = batch
    summary = batch["summary"].copy()
    summary.insert(0, "population_setting", population)
    summary.insert(0, "scenario", scenario)
    summary["batch_complete"] = batch["batch_complete"]
    summaries.append(summary)
    display(summary)
    if not batch["batch_complete"]:
        print("Some files failed. See failed_files.csv in:", output_dir)

combined_summary = pd.concat(summaries, ignore_index=True)
combined_summary.to_csv(RESULT_ROOT / "all_two_cluster_summaries.csv", index=False)
print("\nFour batch calls finished. Results:", RESULT_ROOT)
display(combined_summary)


Scenario_1, Constant: 100 datasets available.
Scenario_1, Original_Population: 100 datasets available.
Scenario_2, Constant: 100 datasets available.
Scenario_2, Original_Population: 100 datasets available.

RUNNING Scenario_1: Constant
Output: C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data\SVTT_fixed_R0_q25_B100_d399284a27\Scenario_1\Constant
SAVED-DATA SVTT BATCH WITH METHOD CHECKPOINTS
Input directory : C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data\Two_Cluster\Scenario_1\Constant
Files selected  : 100
Study years     : 1980--2000
True cluster(s) : [[3, 15, 31], [12, 21, 32]]
n_max           : 2
Candidates      : 232
Window limit    : population share <= 25%
Largest window  : 24.242424%
Expected counts : population_it * pooled R0 (all locations and times)
Monte Carlo     : multinomial by time; fitted county intercepts
Bmc             : 100
Methods         : ['linear', 'quadratic', 'poly3', 'poly4', 'spline3', 'spline4']
Population      : read from each CSV county-year row
Checkpo

,scenario,population_setting,method,n_sim,n_rejected,power,mean_min_p_value,mean_T_obs_max,mean_n_selected,mean_n_significant,...,specificity,precision,jaccard,jaccard_rej,rand_index,adjusted_rand_index,purity,accuracy,exact_recovery,batch_complete
0,Scenario_1,Constant,linear,100,7,0.07,0.464950,3.548247,2.0,0.07,...,0.990741,0.015357,0.008882,0.126881,0.687917,0.003175,0.813030,0.813030,0.00,True
1,Scenario_1,Constant,poly3,100,100,1.00,0.009901,302.047186,2.0,2.00,...,0.752222,0.368995,0.318799,0.318799,0.607955,0.183120,0.733333,0.733333,0.00,True
2,Scenario_1,Constant,poly4,100,100,1.00,0.009901,301.520121,2.0,2.00,...,0.999259,0.997143,0.995476,0.995476,0.998182,0.995764,0.999091,0.999091,0.97,True
3,Scenario_1,Constant,quadratic,100,13,0.13,0.397426,5.333679,2.0,0.17,...,0.982963,0.033389,0.017527,0.134820,0.684773,0.008068,0.809091,0.809091,0.00,True
4,Scenario_1,Constant,spline3,100,100,1.00,0.009901,334.165073,2.0,2.00,...,0.748519,0.365551,0.318308,0.318308,0.605909,0.180765,0.730909,0.730909,0.00,True
5,Scenario_1,Constant,spline4,100,100,1.00,0.009901,326.774946,2.0,2.00,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00,True



RUNNING Scenario_1: Original_Population
Output: C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data\SVTT_fixed_R0_q25_B100_d399284a27\Scenario_1\Original_Population
SAVED-DATA SVTT BATCH WITH METHOD CHECKPOINTS
Input directory : C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data\Two_Cluster\Scenario_1\Original_Population
Files selected  : 100
Study years     : 1980--2000
True cluster(s) : [[3, 15, 31], [12, 21, 32]]
n_max           : 2
Candidates      : 258
Window limit    : population share <= 25%
Largest window  : 24.815242%
Expected counts : population_it * pooled R0 (all locations and times)
Monte Carlo     : multinomial by time; fitted county intercepts
Bmc             : 100
Methods         : ['linear', 'quadratic', 'poly3', 'poly4', 'spline3', 'spline4']
Population      : read from each CSV county-year row
Checkpoint      : after every method and every input file

------------------------------------------------------------------------------------------
FILE 001/100: sim_001.csv
----

,scenario,population_setting,method,n_sim,n_rejected,power,mean_min_p_value,mean_T_obs_max,mean_n_selected,mean_n_significant,...,specificity,precision,jaccard,jaccard_rej,rand_index,adjusted_rand_index,purity,accuracy,exact_recovery,batch_complete
0,Scenario_1,Original_Population,linear,100,10,0.10,0.410198,3.493488,2.0,0.11,...,0.980370,0.036345,0.022297,0.222969,0.685758,0.010240,0.811212,0.811212,0.00,True
1,Scenario_1,Original_Population,poly3,100,100,1.00,0.009901,1070.193832,2.0,2.00,...,0.712963,0.314345,0.255468,0.255468,0.565758,0.098522,0.688788,0.688788,0.00,True
2,Scenario_1,Original_Population,poly4,100,100,1.00,0.009901,1070.203804,2.0,2.00,...,0.826296,0.581744,0.543111,0.543111,0.734432,0.447469,0.811818,0.811818,0.37,True
3,Scenario_1,Original_Population,quadratic,100,7,0.07,0.341485,5.143267,2.0,0.07,...,0.991111,0.024940,0.010141,0.144867,0.688750,0.005144,0.813939,0.813939,0.00,True
4,Scenario_1,Original_Population,spline3,100,100,1.00,0.009901,1182.973628,2.0,2.00,...,0.710741,0.313941,0.256352,0.256352,0.564962,0.098083,0.687879,0.687879,0.00,True
5,Scenario_1,Original_Population,spline4,100,100,1.00,0.009901,1154.982856,2.0,2.00,...,0.942222,0.862453,0.847738,0.847738,0.911780,0.816209,0.937576,0.937576,0.78,True



RUNNING Scenario_2: Constant
Output: C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data\SVTT_fixed_R0_q25_B100_d399284a27\Scenario_2\Constant
SAVED-DATA SVTT BATCH WITH METHOD CHECKPOINTS
Input directory : C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data\Two_Cluster\Scenario_2\Constant
Files selected  : 100
Study years     : 1980--2000
True cluster(s) : [[3, 15, 31], [12, 21, 32]]
n_max           : 2
Candidates      : 232
Window limit    : population share <= 25%
Largest window  : 24.242424%
Expected counts : population_it * pooled R0 (all locations and times)
Monte Carlo     : multinomial by time; fitted county intercepts
Bmc             : 100
Methods         : ['linear', 'quadratic', 'poly3', 'poly4', 'spline3', 'spline4']
Population      : read from each CSV county-year row
Checkpoint      : after every method and every input file

------------------------------------------------------------------------------------------
FILE 001/100: sim_001.csv
-------------------------------------

,scenario,population_setting,method,n_sim,n_rejected,power,mean_min_p_value,mean_T_obs_max,mean_n_selected,mean_n_significant,...,specificity,precision,jaccard,jaccard_rej,rand_index,adjusted_rand_index,purity,accuracy,exact_recovery,batch_complete
0,Scenario_2,Constant,linear,100,12,0.12,0.369901,3.918089,2.0,0.16,...,0.982593,0.042104,0.022465,0.187205,0.686023,0.013816,0.810303,0.810303,0.00,True
1,Scenario_2,Constant,poly3,100,100,1.00,0.009901,28.967644,2.0,2.00,...,0.986296,0.955653,0.946764,0.946764,0.975379,0.945354,0.986970,0.986970,0.73,True
2,Scenario_2,Constant,poly4,100,100,1.00,0.009901,298.335518,2.0,2.00,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00,True
3,Scenario_2,Constant,quadratic,100,9,0.09,0.342079,5.339459,2.0,0.11,...,0.982963,0.018175,0.011750,0.130556,0.683523,0.002087,0.808182,0.808182,0.00,True
4,Scenario_2,Constant,spline3,100,100,1.00,0.009901,32.520884,2.0,2.00,...,0.989259,0.963955,0.957288,0.957288,0.980720,0.956916,0.990000,0.990000,0.77,True
5,Scenario_2,Constant,spline4,100,100,1.00,0.009901,337.316022,2.0,2.00,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00,True



RUNNING Scenario_2: Original_Population
Output: C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data\SVTT_fixed_R0_q25_B100_d399284a27\Scenario_2\Original_Population
SAVED-DATA SVTT BATCH WITH METHOD CHECKPOINTS
Input directory : C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data\Two_Cluster\Scenario_2\Original_Population
Files selected  : 100
Study years     : 1980--2000
True cluster(s) : [[3, 15, 31], [12, 21, 32]]
n_max           : 2
Candidates      : 258
Window limit    : population share <= 25%
Largest window  : 24.815242%
Expected counts : population_it * pooled R0 (all locations and times)
Monte Carlo     : multinomial by time; fitted county intercepts
Bmc             : 100
Methods         : ['linear', 'quadratic', 'poly3', 'poly4', 'spline3', 'spline4']
Population      : read from each CSV county-year row
Checkpoint      : after every method and every input file

------------------------------------------------------------------------------------------
FILE 001/100: sim_001.csv
----

,scenario,population_setting,method,n_sim,n_rejected,power,mean_min_p_value,mean_T_obs_max,mean_n_selected,mean_n_significant,...,specificity,precision,jaccard,jaccard_rej,rand_index,adjusted_rand_index,purity,accuracy,exact_recovery,batch_complete
0,Scenario_2,Original_Population,linear,100,23,0.23,0.267030,4.403366,2.0,0.26,...,0.968889,0.094167,0.050596,0.219985,0.684091,0.035168,0.806667,0.806667,0.00,True
1,Scenario_2,Original_Population,poly3,100,100,1.00,0.009901,70.868954,2.0,1.96,...,0.976667,0.932048,0.900357,0.900357,0.954280,0.896459,0.974545,0.974545,0.62,True
2,Scenario_2,Original_Population,poly4,100,100,1.00,0.009901,849.890904,2.0,2.00,...,0.999630,0.998571,0.998571,0.998571,0.999394,0.998623,0.999697,0.999697,0.99,True
3,Scenario_2,Original_Population,quadratic,100,16,0.16,0.338614,5.270017,2.0,0.16,...,0.978148,0.071229,0.038649,0.241559,0.687689,0.027286,0.811515,0.811515,0.00,True
4,Scenario_2,Original_Population,spline3,100,100,1.00,0.009901,80.597043,2.0,1.99,...,0.978889,0.936857,0.918643,0.918643,0.962235,0.915397,0.979091,0.979091,0.64,True
5,Scenario_2,Original_Population,spline4,100,100,1.00,0.009901,961.027238,2.0,2.00,...,0.999630,0.998571,0.998571,0.998571,0.999394,0.998623,0.999697,0.999697,0.99,True



Four batch calls finished. Results: C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data\SVTT_fixed_R0_q25_B100_d399284a27


,scenario,population_setting,method,n_sim,n_rejected,power,mean_min_p_value,mean_T_obs_max,mean_n_selected,mean_n_significant,...,specificity,precision,jaccard,jaccard_rej,rand_index,adjusted_rand_index,purity,accuracy,exact_recovery,batch_complete
0,Scenario_1,Constant,linear,100,7,0.07,0.464950,3.548247,2.0,0.07,...,0.990741,0.015357,0.008882,0.126881,0.687917,0.003175,0.813030,0.813030,0.00,True
1,Scenario_1,Constant,poly3,100,100,1.00,0.009901,302.047186,2.0,2.00,...,0.752222,0.368995,0.318799,0.318799,0.607955,0.183120,0.733333,0.733333,0.00,True
2,Scenario_1,Constant,poly4,100,100,1.00,0.009901,301.520121,2.0,2.00,...,0.999259,0.997143,0.995476,0.995476,0.998182,0.995764,0.999091,0.999091,0.97,True
3,Scenario_1,Constant,quadratic,100,13,0.13,0.397426,5.333679,2.0,0.17,...,0.982963,0.033389,0.017527,0.134820,0.684773,0.008068,0.809091,0.809091,0.00,True
4,Scenario_1,Constant,spline3,100,100,1.00,0.009901,334.165073,2.0,2.00,...,0.748519,0.365551,0.318308,0.318308,0.605909,0.180765,0.730909,0.730909,0.00,True
5,Scenario_1,Constant,spline4,100,100,1.00,0.009901,326.774946,2.0,2.00,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00,True
6,Scenario_1,Original_Population,linear,100,10,0.10,0.410198,3.493488,2.0,0.11,...,0.980370,0.036345,0.022297,0.222969,0.685758,0.010240,0.811212,0.811212,0.00,True
7,Scenario_1,Original_Population,poly3,100,100,1.00,0.009901,1070.193832,2.0,2.00,...,0.712963,0.314345,0.255468,0.255468,0.565758,0.098522,0.688788,0.688788,0.00,True
8,Scenario_1,Original_Population,poly4,100,100,1.00,0.009901,1070.203804,2.0,2.00,...,0.826296,0.581744,0.543111,0.543111,0.734432,0.447469,0.811818,0.811818,0.37,True
9,Scenario_1,Original_Population,quadratic,100,7,0.07,0.341485,5.143267,2.0,0.07,...,0.991111,0.024940,0.010141,0.144867,0.688750,0.005144,0.813939,0.813939,0.00,True


In [1]:
"""Load saved SVTT batches and add cluster-level recovery metrics.

Run this file in a Jupyter cell with %run, or run it as a Python script.
The Monte Carlo scans are not rerun. Original CSVs are kept intact.

The original scanner's exact_recovery, adjusted_rand_index and purity
describe binary county-union labels. New, separately named columns below
describe the two separate clusters and use the HDSpatialScan convention.
"""
from pathlib import Path
import ast
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score

RESULT_ROOT = Path(
    r"C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data"
    r"\SVTT_fixed_R0_q25_B100_d399284a27"
)
TRUE_CLUSTERS = (frozenset((3, 15, 31)), frozenset((12, 21, 32)))
COUNTIES = tuple(range(1, 34))
METHODS = ("linear", "quadratic", "poly3", "poly4", "spline3", "spline4")
SCENARIOS = ("Scenario_1", "Scenario_2")
POPULATIONS = ("Constant", "Original_Population")
NEW_METRICS = (
    "matched_jaccard", "both_clusters_detected", "exact_cluster_recovery",
    "exact_union_recovery", "partition_ari", "cluster_purity",
)
VERSION = "2026-10-01-fixed-R0-population-v1"


def parse_bool(value):
    """Avoid interpreting the CSV string 'False' as Python True."""
    text = str(value).strip().lower()
    if text in ("true", "1", "1.0"):
        return True
    if text in ("false", "0", "0.0"):
        return False
    raise ValueError(f"Invalid saved Boolean: {value!r}")


def parse_cluster(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return frozenset()
    if isinstance(value, str):
        text = value.strip()
        if text.lower() in ("", "none", "nan"):
            return frozenset()
        value = ast.literal_eval(text)
    if not isinstance(value, (list, tuple, set, np.ndarray)):
        raise ValueError(f"Unexpected saved cluster: {value!r}")
    ids = [int(x) for x in value]
    if any(float(x) != int(x) for x in value) or not set(ids).issubset(COUNTIES):
        raise ValueError(f"Invalid county IDs: {value!r}")
    return frozenset(ids)


def jaccard(a, b):
    return len(a & b) / len(a | b) if a or b else 1.0


def recovery_metrics(detected):
    """Unmatched true clusters score zero; use distinct detected clusters."""
    if not detected:
        matched = (0.0, 0.0)
    elif len(detected) == 1:
        matched = (max(jaccard(t, detected[0]) for t in TRUE_CLUSTERS), 0.0)
    else:
        # j2 outer/j1 inner also reproduces the R code's tie-breaking order.
        pairs = [(j1, j2) for j2 in range(len(detected))
                 for j1 in range(len(detected)) if j1 != j2]
        j1, j2 = max(pairs, key=lambda p:
                     jaccard(TRUE_CLUSTERS[0], detected[p[0]]) +
                     jaccard(TRUE_CLUSTERS[1], detected[p[1]]))
        matched = (jaccard(TRUE_CLUSTERS[0], detected[j1]),
                   jaccard(TRUE_CLUSTERS[1], detected[j2]))

    true_union = frozenset().union(*TRUE_CLUSTERS)
    detected_union = frozenset().union(*detected)
    canonical = lambda clusters: sorted(tuple(sorted(c)) for c in clusters)
    out = {
        "matched_jaccard": sum(matched) / 2,
        "both_clusters_detected": int(all(x > 0 for x in matched)),
        "exact_cluster_recovery": int(canonical(detected) == canonical(TRUE_CLUSTERS)),
        "exact_union_recovery": int(detected_union == true_union),
        "partition_ari": np.nan,
        "cluster_purity": np.nan,
    }
    if sum(map(len, detected)) != len(detected_union):
        return out  # Overlapping clusters do not define a partition.
    def labels(clusters):
        return [next((k for k, c in enumerate(clusters, 1) if county in c), 0)
                for county in COUNTIES]
    truth_labels = labels(TRUE_CLUSTERS)
    detected_labels = labels(detected)
    tab = pd.crosstab(pd.Series(truth_labels), pd.Series(detected_labels))
    out["partition_ari"] = adjusted_rand_score(truth_labels, detected_labels)
    out["cluster_purity"] = tab.max(axis=0).sum() / len(COUNTIES)
    return out


def calculate_folder(folder):
    summary_file = folder / "power_and_cluster_metrics_summary.csv"
    if not summary_file.is_file():
        raise FileNotFoundError(summary_file)
    raw_file = folder / "all_100_files_mc_results.csv"
    if not raw_file.is_file():
        raw_file = folder / "all_completed_files_mc_results.csv"
    if not raw_file.is_file():
        raise FileNotFoundError(f"No saved rank-level results in {folder}")
    summary = pd.read_csv(summary_file)
    raw = pd.read_csv(raw_file)
    required = {"sim", "method", "rank", "converged", "significant", "best_cluster"}
    if missing := required - set(raw.columns):
        raise ValueError(f"{raw_file}: missing columns {sorted(missing)}")
    if raw[list(required - {"best_cluster"})].isna().any().any():
        raise ValueError(f"Missing identifying fields or flags in {raw_file}")
    if raw.duplicated(["sim", "method", "rank"]).any():
        raise ValueError(f"Duplicate simulation/method/rank rows in {raw_file}")
    if "implementation_version" in raw and set(raw.implementation_version) != {VERSION}:
        raise ValueError(f"Unexpected scanner version in {raw_file}")
    if "expected_count_definition" in raw and set(raw.expected_count_definition) != {"population_it * pooled_R0"}:
        raise ValueError(f"Unexpected expected-count definition in {raw_file}")
    if not set(raw.method).issubset(METHODS):
        raise ValueError(f"Unexpected methods in {raw_file}")
    if raw["rank"].max() > 2 or raw["rank"].min() < 1:
        raise ValueError("These folders should contain at most two ranked clusters.")
    for column in ("converged", "significant"):
        raw[column] = raw[column].map(parse_bool)
    if (raw.significant & ~raw.converged).any():
        raise ValueError("A significant rank has a failed fit.")
    rows = []
    for (sim, method), group in raw.groupby(["sim", "method"], sort=False):
        selected = group.loc[group.converged & group.significant].sort_values("rank")
        detected = [parse_cluster(x) for x in selected.best_cluster]
        if any(not c for c in detected):
            raise ValueError(f"Missing membership for significant {method}, sim {sim}")
        rows.append({"sim": sim, "method": method, **recovery_metrics(detected)})
    per_sim = pd.DataFrame(rows)
    if per_sim.empty:
        raise ValueError(f"No saved replicates in {raw_file}")
    means = per_sim.groupby("method")[list(NEW_METRICS)].mean()
    counts = per_sim.groupby("method").size()
    if "method" not in summary or summary.method.duplicated().any():
        raise ValueError(f"Invalid method identifiers in {summary_file}")
    if set(summary.method) != set(counts.index):
        raise ValueError("Saved summary and replicate methods differ.")
    if "n_sim" in summary and not np.array_equal(summary.n_sim.to_numpy(), summary.method.map(counts).to_numpy()):
        raise ValueError("Saved summary and replicate counts differ.")
    audit = []
    for metric in NEW_METRICS:
        present = metric in summary.columns
        if not present:
            summary[metric] = np.nan
        summary[metric] = pd.to_numeric(summary[metric], errors="coerce")
        missing = ~np.isfinite(summary[metric])
        summary.loc[missing, metric] = summary.loc[missing, "method"].map(means[metric])
        audit.append({"metric": metric, "present_in_saved_summary": present,
                      "method_values_filled": int(missing.sum()),
                      "still_missing": int(summary[metric].isna().sum())})
    summary["n_recovery_sim"] = summary.method.map(counts)
    if per_sim.partition_ari.isna().any():
        warnings.warn(f"{folder}: overlapping clusters; partition ARI/purity remain NA.")
    return summary, per_sim, pd.DataFrame(audit)


def enrich_all(result_root=RESULT_ROOT):
    result_root = Path(result_root)
    save_root = result_root / "recovery_metrics"
    combined, replicate_frames, audits = [], [], []
    for scenario in SCENARIOS:
        for population in POPULATIONS:
            summary, per_sim, audit = calculate_folder(result_root / scenario / population)
            for frame in (summary, per_sim, audit):
                frame.insert(0, "population_setting", population)
                frame.insert(0, "scenario", scenario)
            output = save_root / scenario / population
            output.mkdir(parents=True, exist_ok=True)
            summary.to_csv(output / "summary_with_recovery_metrics.csv", index=False)
            per_sim.to_csv(output / "replicate_recovery_metrics.csv", index=False)
            audit.to_csv(output / "metric_check.csv", index=False)
            combined.append(summary)
            replicate_frames.append(per_sim)
            audits.append(audit)
    complete = pd.concat(combined, ignore_index=True)
    complete["_method_order"] = complete.method.map({m: i for i, m in enumerate(METHODS)})
    complete = complete.sort_values(["scenario", "population_setting", "_method_order"]).drop(columns="_method_order").reset_index(drop=True)
    metric_check = pd.concat(audits, ignore_index=True)
    complete.to_csv(save_root / "all_two_cluster_summaries_with_recovery.csv", index=False)
    pd.concat(replicate_frames, ignore_index=True).to_csv(save_root / "all_replicate_recovery_metrics.csv", index=False)
    metric_check.to_csv(save_root / "all_metric_checks.csv", index=False)
    return complete, metric_check


if __name__ == "__main__":
    svtt_complete_results, metric_check = enrich_all()
    svtt_scenario1 = svtt_complete_results.loc[svtt_complete_results.scenario == "Scenario_1"].copy()
    svtt_scenario2 = svtt_complete_results.loc[svtt_complete_results.scenario == "Scenario_2"].copy()
    panel_b_columns = ["scenario", "population_setting", "method", "n_recovery_sim",
                       "exact_cluster_recovery", "matched_jaccard", "partition_ari",
                       "cluster_purity", "both_clusters_detected", "exact_union_recovery"]
    panel_b_scenario1 = svtt_scenario1[panel_b_columns].copy()
    panel_b_scenario2 = svtt_scenario2[panel_b_columns].copy()
    print("\nMetric presence and calculation check:\n")
    print(metric_check.to_string(index=False))
    for title, frame in (("SCENARIO 1: PANEL B", panel_b_scenario1),
                         ("SCENARIO 2: PANEL B", panel_b_scenario2)):
        print(f"\n{title}\n")
        print(frame.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("\nSaved:", RESULT_ROOT / "recovery_metrics")



Metric presence and calculation check:

  scenario  population_setting                 metric  present_in_saved_summary  method_values_filled  still_missing
Scenario_1            Constant        matched_jaccard                     False                     6              0
Scenario_1            Constant both_clusters_detected                     False                     6              0
Scenario_1            Constant exact_cluster_recovery                     False                     6              0
Scenario_1            Constant   exact_union_recovery                     False                     6              0
Scenario_1            Constant          partition_ari                     False                     6              0
Scenario_1            Constant         cluster_purity                     False                     6              0
Scenario_1 Original_Population        matched_jaccard                     False                     6              0
Scenario_1 Original_Pop

## Single Cluster | Constant Population

In [2]:
"""Run nine constant-population SVTT batches in a local Jupyter notebook.

Single_County, Double_County, and Triple_County each run Res1, Res2, and Res3.
Every batch has its own checkpoints and six-method summary. The master CSV
concatenates these labelled summaries; it does not pool different scenarios.

First run the updated svtt_fixed_r0.py definition cell (or %run -i its path).
Then paste this script into another cell, or run:
    %run -i ./run_constant_population_all_cluster_sizes.py
"""

from pathlib import Path
import gc
import hashlib
import json
import re
import traceback
from types import SimpleNamespace

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


# Local Windows data root from your existing two-cluster notebook.
SIM_ROOT = Path(r"C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data")
# Expected layout: Sim_Data/Single_Cluster/<county size>/Constant/<Res1-Res3>.
DATA_ROOT = SIM_ROOT / "Single_Cluster"
N_FILES = 100
BMC = 100
ALPHA = 0.05
MAX_SIZE_PERCENT = 25.0
MIN_SIZE = 1
STEP = 1
SEED = 218720
RESUME = True
METHOD_NAMES = ("linear", "quadratic", "poly3", "poly4", "spline3", "spline4")
# Accept sim_1.csv, sim_01.csv, sim_001.csv, and upper-case CSV extensions.
FILE_PATTERN = "sim_[0-9]*.[cC][sS][vV]"

POPULATION_SETTING = "Constant"
RESIDUAL_SCENARIOS = ("Res1", "Res2", "Res3")
CLUSTER_CONFIGURATIONS = {
    "Single_County": [15],
    "Double_County": [15, 31],
    "Triple_County": [3, 15, 31],
}

# Optional overrides if a batch is stored elsewhere. Keys are
# (configuration, residual), and paths are absolute or relative to SIM_ROOT.
INPUT_DIR_OVERRIDES = {}

# Each configuration contains ONE spatial cluster of one, two, or three
# counties. N_MAX is the number of selected clusters, not the county count.
N_MAX = 1
RUNNER_VERSION = "2026-10-04-local-constant-population-single-double-triple-v1"
REQUIRED_IMPLEMENTATION_VERSION = "2026-10-01-fixed-R0-population-v1"


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def folder_diagnostics(folder):
    """Include observed folder contents in an input-discovery error."""
    lines = [f"Folder: {folder}", f"Directory exists: {folder.is_dir()}"]
    location = folder if folder.is_dir() else folder.parent
    if location.is_dir():
        entries = sorted(location.iterdir(), key=lambda p: p.name)
        sample = [p.name + ("/" if p.is_dir() else "") for p in entries[:30]]
        lines.append(f"Contents of {location} (first 30): {sample or '<empty>'}")
    if folder.is_dir():
        csv_samples = []
        for path in folder.rglob("*"):
            if path.is_file() and path.suffix.lower() == ".csv":
                csv_samples.append(str(path.relative_to(folder)))
                if len(csv_samples) == 10:
                    break
        lines.append(f"CSV examples including subfolders: {csv_samples or '<none>'}")
    return "\n".join(lines)


def discover_batch_files(folder):
    """Resolve direct inputs or one unambiguous nested simulation directory.

    Different directories are never combined to reach N_FILES. In particular,
    discovery stays within the requested configuration/population/scenario.
    """
    if not folder.is_dir():
        raise FileNotFoundError(
            "Input folder not found.\n" + folder_diagnostics(folder)
        )
    direct = sorted(
        (p for p in folder.glob(FILE_PATTERN) if p.is_file()),
        key=svtt._natural_path_key,
    )
    if not direct:
        nested_groups = {}
        for path in folder.rglob(FILE_PATTERN):
            if path.is_file():
                nested_groups.setdefault(path.parent, []).append(path)
        if len(nested_groups) == 1:
            folder, nested = next(iter(nested_groups.items()))
            direct = sorted(nested, key=svtt._natural_path_key)
            print(f"Simulation files are in a subfolder; using: {folder}")
        elif len(nested_groups) > 1:
            candidates = "\n".join(
                f"  {directory}: {len(paths)} matching files"
                for directory, paths in sorted(nested_groups.items())
            )
            raise ValueError(
                "Simulation files occur in multiple subfolders. Set the "
                "appropriate INPUT_DIR_OVERRIDES entry to the intended folder.\n"
                + candidates + "\n" + folder_diagnostics(folder)
            )
    if len(direct) < N_FILES:
        raise ValueError(
            f"Found {len(direct)} files matching {FILE_PATTERN!r}; need {N_FILES}.\n"
            + folder_diagnostics(folder)
            + "\nCheck the CSV filenames and directory layout. If files have "
            "other names, set FILE_PATTERN to their dataset filename pattern."
        )
    selected = direct[:N_FILES]
    if FILE_PATTERN == "sim_[0-9]*.[cC][sS][vV]":
        non_datasets = [
            p.name for p in selected
            if re.fullmatch(r"sim_[0-9]+\.[cC][sS][vV]", p.name) is None
        ]
        if non_datasets:
            raise ValueError(
                "The simulation pattern also matched other files: "
                f"{non_datasets}. Keep simulation inputs in a separate directory "
                "or choose a more specific FILE_PATTERN.\n"
                + folder_diagnostics(folder)
            )
    return folder, direct


# Reuse the SVTT definitions loaded in the notebook, as in your pasted runner.
required_names = (
    "run_100_saved_files", "select_methods", "_natural_path_key",
    "_normalise_saved_dataset_columns", "infer_true_clusters_from_saved_data",
    "_canonical_clusters", "_atomic_write_csv", "IMPLEMENTATION_VERSION",
    "EXPECTED_COUNT_DEFINITION",
)
missing = [name for name in required_names if name not in globals()]
if not missing:
    svtt = SimpleNamespace(**{name: globals()[name] for name in required_names})
elif "svtt" not in globals() or any(not hasattr(svtt, name) for name in required_names):
    raise RuntimeError(
        "Run the updated svtt_fixed_r0.py definition cell first, or use "
        "%run -i with its local path. Then use %run -i for this runner too. "
        "Missing definitions: " + ", ".join(missing)
    )
if getattr(svtt, "IMPLEMENTATION_VERSION", None) != REQUIRED_IMPLEMENTATION_VERSION:
    raise RuntimeError("Run the updated svtt_fixed_r0.py first; the loaded scanner is old.")
methods = svtt.select_methods(METHOD_NAMES)

if "COORD_PATH" not in globals():
    COORD_PATH = Path(
        r"C:\Users\ansarims\Desktop\RK_Summer1_2026"
        r"\Code\SVTT_24Aug2026\nm_county_coordinates.txt"
    )
COORD_PATH = Path(COORD_PATH).expanduser().resolve()
if not COORD_PATH.is_file():
    raise FileNotFoundError(f"Coordinate file not found: {COORD_PATH}")
SIM_ROOT = SIM_ROOT.expanduser().resolve()
DATA_ROOT = DATA_ROOT.expanduser().resolve()

# Build all nine blocks. Only Constant folders are selected.
blocks = [
    (configuration, residual)
    for configuration in CLUSTER_CONFIGURATIONS
    for residual in RESIDUAL_SCENARIOS
]

# Validate all folders and first-file truth before starting any long batch.
# Hash the selected input files so resume cannot reuse different datasets.
input_batches = []
plan_rows = []
resolved_input_dirs = {}
for configuration, residual in blocks:
    true_cluster = CLUSTER_CONFIGURATIONS[configuration]
    default_dir = DATA_ROOT / configuration / POPULATION_SETTING / residual
    folder = Path(INPUT_DIR_OVERRIDES.get((configuration, residual), default_dir))
    folder = folder.expanduser()
    if not folder.is_absolute():
        folder = SIM_ROOT / folder
    folder, available = discover_batch_files(folder.resolve())
    resolved_input_dirs[(configuration, residual)] = folder
    selected = available[:N_FILES]
    first_data = svtt._normalise_saved_dataset_columns(
        pd.read_csv(selected[0]), source_path=selected[0]
    )
    inferred_truth = svtt.infer_true_clusters_from_saved_data(first_data)
    if inferred_truth is not None and (
        svtt._canonical_clusters(inferred_truth)
        != svtt._canonical_clusters(true_cluster)
    ):
        raise ValueError(
            f"{selected[0]}: saved truth is {inferred_truth}; "
            f"{configuration} is configured as {true_cluster}. "
            "Check CLUSTER_CONFIGURATIONS before calculating recovery metrics."
        )
    missing_counties = set(true_cluster) - set(first_data["county"])
    if missing_counties:
        raise ValueError(
            f"{selected[0]}: true-cluster counties missing: "
            f"{sorted(missing_counties)}"
        )
    file_fingerprints = [(p.name, sha256_file(p)) for p in selected]
    input_digest = hashlib.sha256(
        json.dumps(file_fingerprints).encode("utf-8")
    ).hexdigest()
    input_batches.append(dict(
        cluster_configuration=configuration,
        population_setting=POPULATION_SETTING,
        residual_scenario=residual,
        true_cluster=true_cluster,
        input_dir=str(folder),
        selected_files=[p.name for p in selected],
        input_sha256=input_digest,
    ))
    plan_rows.append(dict(
        cluster_configuration=configuration,
        population_setting=POPULATION_SETTING,
        residual_scenario=residual,
        true_cluster=json.dumps(true_cluster),
        n_true_counties=len(true_cluster),
        available_files=len(available),
        requested_files=N_FILES,
        input_dir=str(folder),
    ))
    print(
        f"{configuration}/{POPULATION_SETTING}/{residual}: "
        f"{len(available)} datasets available; true cluster = {true_cluster}."
    )

settings = dict(
    runner_version=RUNNER_VERSION,
    implementation_version=svtt.IMPLEMENTATION_VERSION,
    expected_count_definition=svtt.EXPECTED_COUNT_DEFINITION,
    coordinate_sha256=sha256_file(COORD_PATH),
    n_files=N_FILES, Bmc=BMC, alpha=ALPHA,
    max_size_percent=MAX_SIZE_PERCENT, min_size=MIN_SIZE,
    step=STEP, seed=SEED, n_max=N_MAX, n_jobs=1,
    methods=METHOD_NAMES,
    file_pattern=FILE_PATTERN,
    population_setting=POPULATION_SETTING,
    cluster_configurations=CLUSTER_CONFIGURATIONS,
    residual_scenarios=RESIDUAL_SCENARIOS,
    input_batches=input_batches,
)
tag = hashlib.sha256(
    json.dumps(settings, sort_keys=True).encode("utf-8")
).hexdigest()[:10]
RESULT_ROOT = (
    SIM_ROOT / "Results"
    / f"Constant_All_Cluster_Sizes_fixed_R0_q{MAX_SIZE_PERCENT:g}_B{BMC}_{tag}"
)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
(RESULT_ROOT / "run_settings.json").write_text(
    json.dumps(settings, indent=2), encoding="utf-8"
)
svtt._atomic_write_csv(pd.DataFrame(plan_rows), RESULT_ROOT / "batch_plan.csv")
print("\nSimulation root:", SIM_ROOT)
print("Single-cluster inputs:", DATA_ROOT)
print("Coordinates:", COORD_PATH)
print("Results:", RESULT_ROOT)
print(f"Constant-population batches: {len(blocks)}")

constant_population_batch_results = {}
summaries = []
statuses = []
combined_summary = pd.DataFrame()
method_order = {name: index for index, name in enumerate(METHOD_NAMES)}

for configuration, residual in blocks:
    true_cluster = CLUSTER_CONFIGURATIONS[configuration]
    input_dir = resolved_input_dirs[(configuration, residual)]
    output_dir = RESULT_ROOT / configuration / POPULATION_SETTING / residual
    labels = dict(
        cluster_configuration=configuration,
        population_setting=POPULATION_SETTING,
        residual_scenario=residual,
        true_cluster=json.dumps(true_cluster),
        n_true_counties=len(true_cluster),
    )
    print(f"\nRUNNING: {configuration}/{POPULATION_SETTING}/{residual}")

    try:
        batch = svtt.run_100_saved_files(
            data_dir=input_dir, coord_path=COORD_PATH, output_dir=output_dir,
            true_cluster=list(true_cluster), n_files=N_FILES, n_max=N_MAX,
            methods=methods, Bmc=BMC, alpha=ALPHA,
            min_size=MIN_SIZE, max_size_percent=MAX_SIZE_PERCENT,
            step=STEP, seed=SEED, n_jobs=1,
            file_pattern=FILE_PATTERN, resume=RESUME, stop_on_error=False,
        )
        constant_population_batch_results[(configuration, residual)] = batch
        summary = batch["summary"].copy()
        summary = (
            summary.assign(_method_order=summary["method"].map(method_order))
            .sort_values("_method_order").drop(columns="_method_order")
            .reset_index(drop=True)
        )
        for name, value in reversed(list(labels.items())):
            summary.insert(0, name, value)
        summary["batch_complete"] = bool(batch["batch_complete"])
        summaries.append(summary)

        # One separate, labelled six-method table per batch.
        svtt._atomic_write_csv(summary, output_dir / "summary.csv")
        metric_columns = list(labels) + [
            "method", "n_sim", "power", "sensitivity", "specificity", "batch_complete"
        ]
        svtt._atomic_write_csv(
            summary[metric_columns], output_dir / "sensitivity_specificity.csv"
        )
        print(f"\nSUMMARY: {configuration}/{POPULATION_SETTING}/{residual}")
        display(summary[metric_columns])
        statuses.append(dict(
            **labels,
            status="completed" if batch["batch_complete"] else "incomplete",
            completed_files=int(batch["results"]["sim"].nunique()),
            requested_files=N_FILES, output_dir=str(output_dir), error="",
        ))
        error_path = output_dir / "batch_error.txt"
        if batch["batch_complete"] and error_path.exists():
            error_path.unlink()
        if not batch["batch_complete"]:
            print("Some files failed. See:", output_dir / "failed_files.csv")
    except Exception as error:
        output_dir.mkdir(parents=True, exist_ok=True)
        (output_dir / "batch_error.txt").write_text(
            traceback.format_exc(), encoding="utf-8"
        )
        statuses.append(dict(
            **labels, status="failed", completed_files=None,
            requested_files=N_FILES, output_dir=str(output_dir),
            error=f"{type(error).__name__}: {error}",
        ))
        print("BATCH FAILED:", error)
    finally:
        if summaries:
            # Concatenation preserves configuration/scenario labels. There is
            # deliberately no groupby('method') across these different batches.
            combined_summary = pd.concat(summaries, ignore_index=True)
            svtt._atomic_write_csv(
                combined_summary, RESULT_ROOT / "all_constant_population_summaries.csv"
            )
        svtt._atomic_write_csv(
            pd.DataFrame(statuses), RESULT_ROOT / "batch_status.csv"
        )
        gc.collect()

batch_status = pd.DataFrame(statuses)
display(batch_status)
print("\nResults saved in:", RESULT_ROOT)
print("Each configuration/scenario has its own summary.csv and sensitivity_specificity.csv.")
if not (batch_status["status"] == "completed").all():
    print(
        "Some batches are incomplete. Their summaries use completed datasets only. "
        "Resolve the recorded errors and rerun to resume."
    )


ERROR! Session/line number was not unique in database. History logging moved to new session 224
Single_County/Constant/Res1: 100 datasets available; true cluster = [15].
Single_County/Constant/Res2: 100 datasets available; true cluster = [15].
Single_County/Constant/Res3: 100 datasets available; true cluster = [15].
Double_County/Constant/Res1: 100 datasets available; true cluster = [15, 31].
Double_County/Constant/Res2: 100 datasets available; true cluster = [15, 31].
Double_County/Constant/Res3: 100 datasets available; true cluster = [15, 31].
Triple_County/Constant/Res1: 100 datasets available; true cluster = [3, 15, 31].
Triple_County/Constant/Res2: 100 datasets available; true cluster = [3, 15, 31].
Triple_County/Constant/Res3: 100 datasets available; true cluster = [3, 15, 31].

Simulation root: C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data
Single-cluster inputs: C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data\Single_Cluster
Coordinates: C:\Users\ansarims\Desktop\RK_Summer1_2

,cluster_configuration,population_setting,residual_scenario,true_cluster,n_true_counties,method,n_sim,power,sensitivity,specificity,batch_complete
0,Single_County,Constant,Res1,[15],1,linear,100,0.07,0.01,0.994062,True
1,Single_County,Constant,Res1,[15],1,quadratic,100,0.07,0.01,0.992500,True
2,Single_County,Constant,Res1,[15],1,poly3,100,1.00,1.00,1.000000,True
3,Single_County,Constant,Res1,[15],1,poly4,100,1.00,1.00,1.000000,True
4,Single_County,Constant,Res1,[15],1,spline3,100,1.00,1.00,1.000000,True
5,Single_County,Constant,Res1,[15],1,spline4,100,1.00,1.00,1.000000,True



RUNNING: Single_County/Constant/Res2
SAVED-DATA SVTT BATCH WITH METHOD CHECKPOINTS
Input directory : C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data\Single_Cluster\Single_County\Constant\Res2
Files selected  : 100
Study years     : 1980--2000
True cluster(s) : [15]
n_max           : 1
Candidates      : 232
Window limit    : population share <= 25%
Largest window  : 24.242424%
Expected counts : population_it * pooled R0 (all locations and times)
Monte Carlo     : multinomial by time; fitted county intercepts
Bmc             : 100
Methods         : ['linear', 'quadratic', 'poly3', 'poly4', 'spline3', 'spline4']
Population      : read from each CSV county-year row
Checkpoint      : after every method and every input file

------------------------------------------------------------------------------------------
FILE 001/100: sim_001.csv
------------------------------------------------------------------------------------------
  Pooled R0   : 0.01001053391
Complete file checkpoint found;

,cluster_configuration,population_setting,residual_scenario,true_cluster,n_true_counties,method,n_sim,power,sensitivity,specificity,batch_complete
0,Single_County,Constant,Res2,[15],1,linear,100,0.01,0.0,0.999687,True
1,Single_County,Constant,Res2,[15],1,quadratic,100,0.03,0.0,0.996875,True
2,Single_County,Constant,Res2,[15],1,poly3,100,1.00,1.0,1.000000,True
3,Single_County,Constant,Res2,[15],1,poly4,100,1.00,1.0,1.000000,True
4,Single_County,Constant,Res2,[15],1,spline3,100,1.00,1.0,1.000000,True
5,Single_County,Constant,Res2,[15],1,spline4,100,1.00,1.0,1.000000,True



RUNNING: Single_County/Constant/Res3
SAVED-DATA SVTT BATCH WITH METHOD CHECKPOINTS
Input directory : C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data\Single_Cluster\Single_County\Constant\Res3
Files selected  : 100
Study years     : 1980--2000
True cluster(s) : [15]
n_max           : 1
Candidates      : 232
Window limit    : population share <= 25%
Largest window  : 24.242424%
Expected counts : population_it * pooled R0 (all locations and times)
Monte Carlo     : multinomial by time; fitted county intercepts
Bmc             : 100
Methods         : ['linear', 'quadratic', 'poly3', 'poly4', 'spline3', 'spline4']
Population      : read from each CSV county-year row
Checkpoint      : after every method and every input file

------------------------------------------------------------------------------------------
FILE 001/100: sim_001.csv
------------------------------------------------------------------------------------------
  Pooled R0   : 0.009993795094
  Global null : fitted county 

...

,cluster_configuration,population_setting,residual_scenario,true_cluster,n_true_counties,method,n_sim,power,sensitivity,specificity,batch_complete
0,Triple_County,Constant,Res2,"[3, 15, 31]",3,linear,100,0.06,0.013333,0.992333,True
1,Triple_County,Constant,Res2,"[3, 15, 31]",3,quadratic,100,0.02,0.003333,0.997000,True
2,Triple_County,Constant,Res2,"[3, 15, 31]",3,poly3,100,1.00,1.000000,1.000000,True
3,Triple_County,Constant,Res2,"[3, 15, 31]",3,poly4,100,1.00,1.000000,1.000000,True
4,Triple_County,Constant,Res2,"[3, 15, 31]",3,spline3,100,1.00,1.000000,1.000000,True
5,Triple_County,Constant,Res2,"[3, 15, 31]",3,spline4,100,1.00,1.000000,1.000000,True



RUNNING: Triple_County/Constant/Res3
SAVED-DATA SVTT BATCH WITH METHOD CHECKPOINTS
Input directory : C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data\Single_Cluster\Triple_County\Constant\Res3
Files selected  : 100
Study years     : 1980--2000
True cluster(s) : [3, 15, 31]
n_max           : 1
Candidates      : 232
Window limit    : population share <= 25%
Largest window  : 24.242424%
Expected counts : population_it * pooled R0 (all locations and times)
Monte Carlo     : multinomial by time; fitted county intercepts
Bmc             : 100
Methods         : ['linear', 'quadratic', 'poly3', 'poly4', 'spline3', 'spline4']
Population      : read from each CSV county-year row
Checkpoint      : after every method and every input file

------------------------------------------------------------------------------------------
FILE 001/100: sim_001.csv
------------------------------------------------------------------------------------------
  Pooled R0   : 0.01005064935
  Global null : fitted c

,cluster_configuration,population_setting,residual_scenario,true_cluster,n_true_counties,method,n_sim,power,sensitivity,specificity,batch_complete
0,Triple_County,Constant,Res3,"[3, 15, 31]",3,linear,100,0.08,0.010000,0.991667,True
1,Triple_County,Constant,Res3,"[3, 15, 31]",3,quadratic,100,0.07,0.006667,0.994667,True
2,Triple_County,Constant,Res3,"[3, 15, 31]",3,poly3,100,0.09,0.020000,0.990667,True
3,Triple_County,Constant,Res3,"[3, 15, 31]",3,poly4,100,1.00,1.000000,1.000000,True
4,Triple_County,Constant,Res3,"[3, 15, 31]",3,spline3,100,0.07,0.020000,0.993000,True
5,Triple_County,Constant,Res3,"[3, 15, 31]",3,spline4,100,1.00,1.000000,1.000000,True


,cluster_configuration,population_setting,residual_scenario,true_cluster,n_true_counties,status,completed_files,requested_files,output_dir,error
0,Single_County,Constant,Res1,[15],1,completed,100,100,C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Da...,
1,Single_County,Constant,Res2,[15],1,completed,100,100,C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Da...,
2,Single_County,Constant,Res3,[15],1,completed,100,100,C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Da...,
3,Double_County,Constant,Res1,"[15, 31]",2,completed,100,100,C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Da...,
4,Double_County,Constant,Res2,"[15, 31]",2,completed,100,100,C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Da...,
5,Double_County,Constant,Res3,"[15, 31]",2,completed,100,100,C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Da...,
6,Triple_County,Constant,Res1,"[3, 15, 31]",3,completed,100,100,C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Da...,
7,Triple_County,Constant,Res2,"[3, 15, 31]",3,completed,100,100,C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Da...,
8,Triple_County,Constant,Res3,"[3, 15, 31]",3,completed,100,100,C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Da...,



Results saved in: C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data\Results\Constant_All_Cluster_Sizes_fixed_R0_q25_B100_1c00250a2f
Each configuration/scenario has its own summary.csv and sensitivity_specificity.csv.


In [3]:

from pathlib import Path
import pandas as pd
from IPython.display import display

RESULTS_ROOT = Path(
    r"C:\Users\ansarims\Desktop\Sem3_Fall2027\Sim_Data\Results"
    r"\Constant_All_Cluster_Sizes_fixed_R0_q25_B100_1c00250a2f"
)

configurations = ["Single_County", "Double_County", "Triple_County"]
scenarios = ["Res1", "Res2", "Res3"]
methods = ["linear", "quadratic", "poly3", "poly4", "spline3", "spline4"]

frames = []
missing_files = []

for configuration in configurations:
    for scenario in scenarios:
        summary_path = (
            RESULTS_ROOT / configuration / "Constant" / scenario / "summary.csv"
        )

        if not summary_path.is_file():
            missing_files.append(str(summary_path))
            continue

        summary = pd.read_csv(summary_path)
        summary.columns = summary.columns.str.strip().str.lower()

        required = ["method", "power", "sensitivity", "specificity", "precision"]
        missing_columns = [col for col in required if col not in summary.columns]
        if missing_columns:
            raise ValueError(
                f"{summary_path}: missing columns {missing_columns}"
            )

        summary["method"] = summary["method"].astype(str).str.strip().str.lower()
        summary["recall"] = summary["sensitivity"]
        summary["cluster_configuration"] = configuration
        summary["population_setting"] = "Constant"
        summary["residual_scenario"] = scenario

        frames.append(summary[
            [
                "cluster_configuration",
                "population_setting",
                "residual_scenario",
                "method",
                "power",
                "sensitivity",
                "specificity",
                "precision",
                "recall",
            ]
        ])

if not frames:
    raise FileNotFoundError(f"No summary.csv files found under {RESULTS_ROOT}")

df = pd.concat(frames, ignore_index=True)

# Arrange configurations, scenarios and methods in the desired order.
for column, order in [
    ("cluster_configuration", configurations),
    ("residual_scenario", scenarios),
    ("method", methods),
]:
    df[column] = pd.Categorical(df[column], categories=order, ordered=True)

df = df.sort_values(
    ["cluster_configuration", "residual_scenario", "method"]
).reset_index(drop=True)

metrics = ["power", "sensitivity", "specificity", "precision", "recall"]
df[metrics] = df[metrics].apply(pd.to_numeric, errors="raise")

print(f"Loaded {len(frames)}/9 summary files; {len(df)} method rows.")
if missing_files:
    print("\nMissing files:")
    for path in missing_files:
        print(path)

# Display every row with three decimal places.
with pd.option_context("display.max_rows", None):
    display(df.style.format({col: "{:.3f}" for col in metrics}))


Loaded 9/9 summary files; 54 method rows.


,cluster_configuration,population_setting,residual_scenario,method,power,sensitivity,specificity,precision,recall
0,Single_County,Constant,Res1,linear,0.070,0.010,0.994,0.003,0.010
1,Single_County,Constant,Res1,quadratic,0.070,0.010,0.993,0.003,0.010
2,Single_County,Constant,Res1,poly3,1.000,1.000,1.000,1.000,1.000
3,Single_County,Constant,Res1,poly4,1.000,1.000,1.000,1.000,1.000
4,Single_County,Constant,Res1,spline3,1.000,1.000,1.000,1.000,1.000
5,Single_County,Constant,Res1,spline4,1.000,1.000,1.000,1.000,1.000
6,Single_County,Constant,Res2,linear,0.010,0.000,1.000,0.000,0.000
7,Single_County,Constant,Res2,quadratic,0.030,0.000,0.997,0.000,0.000
8,Single_County,Constant,Res2,poly3,1.000,1.000,1.000,1.000,1.000
9,Single_County,Constant,Res2,poly4,1.000,1.000,1.000,1.000,1.000
